# <font color=RED> ROMPEBRECHAS — Hito 2 </font>
### Preparación de datos y primera matriz analítica

Este notebook reproduce el proceso completo **desde las fuentes hasta la matriz analítica**. Parte de la versión
avanzada hasta la fecha del trabajo y agrega lo que pide el Hito 2: cambios respecto al Hito 1, evidencia de granularidad e integración,
transformaciones de variables justificadas (incluidas las que se probaron y se descartaron) y la matriz analítica
documentada.

| Sección | Contenido |
|---|---|
| 0 | Cambios respecto al Hito 1 |
| 1 – 4 | Carga, diagnóstico y limpieza de cada fuente |
| 5 | Integración, granularidad y validación de cruces |
| 6 | EDA sobre la base integrada |
| 7 | Transformación de variables (codificación, discretización, escalamiento) |
| 8 | Matriz analítica: variables, diccionario y exclusiones |
| 9 | Próximo paso |
| 10 | Conclusiones |

**PROBLEMA:**
¿Qué factores institucionales se asocian al **registro** de casos de violencia escolar en los colegios de educación
primaria del Perú, y cómo se relaciona ese registro con la repetición y el retiro escolar, en el período 2024-2025?

**USUARIO:** especialistas de convivencia escolar de las UGEL y DRE, y el equipo del MINEDU a cargo del SíseVe. Son
quienes deciden en qué colegios capacitar, acompañar o supervisar el registro de casos.

**BENEFICIARIOS:** los estudiantes de primaria, en especial los de colegios donde hoy la violencia no se está registrando
y por lo tanto no se atiende.

**OBJETIVO DE ANÁLISIS:** construir una base a nivel colegio que permita (1) identificar qué características
institucionales se asocian a que un colegio registre incidencias de violencia y (2) evaluar si ese registro se
relaciona con la repetición y el retiro. La idea de fondo es poder distinguir un *colegio sin violencia* de un
*colegio sin capacidad de registro*.

**UNIDAD DE ANÁLISIS**

Las cuatro fuentes se llevan a la misma unidad: **el servicio educativo de primaria (colegio)**, identificado por la
llave compuesta `COD_MOD` + `ANEXO`.

*   `Lineal_3AP_1` (Censo Educativo 2025, Módulo I): respuestas del director sobre convivencia escolar, tutoría y ESI. Ya viene a nivel colegio.
*   `Padron_web`: datos administrativos y geográficos. Ya viene a nivel colegio, pero incluye todos los niveles educativos.
*   `Matricula_01` (cédula 3AP, cuadro C201): viene **por colegio y turno**; se sumaron los turnos para llegar a una fila por colegio.
*   `Resultado_2025` (cédula 3BP): viene en **formato largo** (colegio × cuadro × situación); se pivoteó a una fila por colegio.

**Qué representa cada fila**

*   **Base integrada** (`base_final`): un colegio de primaria, con sus respuestas del censo, su ubicación y gestión, su matrícula y su resultado del año escolar.
*   **Matriz analítica** (`matriz`): el mismo colegio, pero solo con las variables que entran al análisis, ya codificadas, derivadas y escaladas.

**FUENTES DE DATOS USADAS:**

*   Censo Educativo 2025 — Módulo I: Matrícula, Docentes y Recursos (MINEDU-ESCALE): https://escale.minedu.gob.pe/uee/-/document_library_display/GMv7/view/10632985
*   Resultado del Ejercicio Educativo 2025 (MINEDU-ESCALE, fuente SIAGIE): mismo repositorio, archivo `73_Resultado_2025.zip`
*   Padrón de Instituciones Educativas (MINEDU-ESCALE): https://escale.minedu.gob.pe/web/inicio/padron-de-iiee

Las bases procesadas están publicadas en Hugging Face: https://huggingface.co/datasets/themasterdrop/rompebrechas-violencia-escolar

Las bases se descargaron en formato `.dbf`, se filtraron al nivel Primaria y se publicaron en formato `.parquet` en el repositorio del equipo en Hugging Face.

## <font color=yellow> 0. Cambios respecto al Hito 1 </font>


| Aspecto | Hito 1 | Hito 2 | Por qué se cambió |
|---|---|---|---|
| Problema | Factores que influyen en los *casos* de violencia | Factores asociados al *registro* de violencia, y su relación con repetición y retiro | Los datos disponibles miden violencia **registrada**, no ocurrida |
| Fuente de violencia | Denuncias del portal SíseVe | `P124B_SI` y `P126B_SI_1` del Censo (a nivel colegio) | SíseVe solo llega a nivel UGEL: cruzarla con colegios producía una falacia ecológica |
| Fuentes nuevas | — | Resultado del Ejercicio 2025 y Matrícula 2025 | Aportan repetición/retiro y el denominador de las tasas |
| Unidad de análisis | Mezcla de colegio y denuncia | Colegio (`COD_MOD` + `ANEXO`) en las cuatro fuentes | Permite una integración 1 a 1 real |
| Llave | `COD_MOD` sin normalizar en una fuente | `normalizar_llave()` en todas | El merge perdía filas en silencio |
| Tipo de unión | `outer` y luego eliminar filas con umbral de NaN | `inner` (censo–padrón) y `left` (matrícula, resultado) | No crear filas sin colegio real |
| Preguntas condicionadas | Imputación con la mediana | 0 estructural o NaN según la pregunta filtro | La mediana asignaba violencia a colegios que declararon cero |
| Indicadores | Conteos absolutos | Tasas por 100 estudiantes y tasa agregada | Comparar colegios de distinto tamaño |
| Transformaciones | No había | Codificación, discretización, log y z-score justificados (sección 7) | Requisito del Hito 2 y base para el análisis posterior |

## <font color=yellow> 0. Preparación del entorno </font>

In [1]:
import io
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.formula.api as smf
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

pd.set_option("display.max_columns", 60)

# Carpeta donde se guardan la base integrada y la matriz analítica
SALIDAS = Path("salidas")
SALIDAS.mkdir(exist_ok=True)

In [2]:
# --- Origen remoto (Hugging Face) --------------------------------------------
REPO_HF = "themasterdrop/rompebrechas-violencia-escolar"
URL_HF = f"https://huggingface.co/datasets/{REPO_HF}/resolve/main/"

def leer_de_hugging_face(nombre_archivo):
    with urllib.request.urlopen(URL_HF + nombre_archivo, timeout=120) as respuesta:
        return pd.read_parquet(io.BytesIO(respuesta.read()))

print("Origen remoto:", URL_HF)

Origen remoto: https://huggingface.co/datasets/themasterdrop/rompebrechas-violencia-escolar/resolve/main/


In [3]:
# --- Normalización de la llave -----------------------------------------------
# COD_MOD tiene 7 caracteres y ANEXO 1. Si una fuente los trae como número y otra
# como texto, el merge falla en silencio y se pierden filas. Por eso todas las
# fuentes pasan por esta función antes de cruzarse.

LLAVE = ["COD_MOD", "ANEXO"]

def normalizar_llave(df):
    df = df.copy()
    df["COD_MOD"] = df["COD_MOD"].astype(str).str.strip().str.zfill(7)
    df["ANEXO"] = df["ANEXO"].astype(str).str.strip()
    return df

## <font color=yellow> 1. Tratamiento de datos — Lineal_3AP_1 (Censo Educativo) </font>

### <font color=skyblue> 1.1 Importación de datos — Lineal_3AP_1 </font>

In [4]:
df_censo = leer_de_hugging_face("censo_3ap_primaria.parquet")
print("Censo 3AP leído desde Hugging Face")

Censo 3AP leído desde Hugging Face


In [5]:
"""
# Lectura original desde el archivo .dbf descargado de ESCALE:
from dbfread import DBF
tabla_f2 = DBF("Lineal_3AP_1.dbf", encoding="latin-1", char_decode_errors="replace")
df_censo = pd.DataFrame(iter(tabla_f2))
"""

# Convertir nombres de columnas a mayúsculas
df_censo.columns = [c.upper() for c in df_censo.columns]

print("Dimensiones:", df_censo.shape)
display(df_censo.head(5))

Dimensiones: (37354, 193)


,COD_MOD,ANEXO,CODLOCAL,NROCED,CEN_EDU,DISTRITO,LOCALIDAD,P104A,P105A,P105A_S_1,P105A_S_2,P105A_S_3,P105A_S_4,P105A_S_5,P105A_S_5O,P106A,P107A,P107A_NO,P107A_NO_O,P108A,P109A_1,P109A_2,P109A_3,P109A_4,P109A_5,P109A_6,P110A,P111A_1,P111A_2,P111A_3,...,P134B,P134B_S1_1,P134B_S1_2,P134B_S1_3,P134B_S1_4,P134B_S1_5,P134B_S1_6,P134B_S1_7,P134B_S1_O,P134B_S2_1,P134B_S2_2,P134B_S2_3,P135B,P135B_S_1,P135B_S_2,P135B_S_3,P135B_S_4,P135B_S_5,P135B_SI_5,P136B,P136B_S_1,P136B_S_2,P136B_S_3,P136B_S_4,P136B_S_4O,NIV_MOD,GES_DEP,AREA_CENSO,CODGEO,CODOOII
0,1720556,0,852373,3AP,SIR ALEXANDER FLEMING,CERCADO,,NO,,,,,,,,NO,,,,NO,X,,,,,,SI,X,,,...,NO,,,,,,,,,,,,SI,X,,,,,,SI,X,X,,,,B0,B4,1,040101,040001
1,0313890,0,073181,3AP,40513 SAN MIGUEL ARCANGEL,ALCA,ANEXO DE CAHUANA,NO,,,,,,,,NO,,,,SI,,,,,,X,SI,X,X,X,...,SI,X,X,X,,,,,,X,,X,SI,X,X,X,X,,,SI,X,X,X,,,B0,A1,2,040802,040009
2,1495092,0,605028,3AP,33421,PILLCO MARCA,CAYHUAYNA,NO,,,,,,,,SI,NO,2,,SI,,,,X,,,SI,X,,X,...,NO,,,,,,,,,,,,NO,,,,,,,NO,,,,,,B0,A1,1,100111,100001
3,1579861,0,651751,3AP,SANTANDER DE PUENTE PIEDRA,PUENTE PIEDRA,SANTA PAULA,NO,,,,,,,,SI,SI,,,SI,,,,,,X,SI,X,,,...,NO,,,,,,,,,,,,NO,,,,,,,NO,,,,,,B0,B4,1,150125,150105
4,1529767,0,553166,3AP,COLEGIO SUPERIOR DE CIENCIAS NEWTON VIANNEY,YURA,,NO,,,,,,,,SI,SI,,,SI,,,,,,X,SI,X,,,...,NO,,,,,,,,,,,,NO,,,,,,,SI,,X,,,,B0,B4,1,040128,040001


### <font color=skyblue> 1.2 Ajuste de columnas — Lineal_3AP_1 </font>

De las 193 variables de la cédula 3AP se seleccionan las de identificación y las
vinculadas a **convivencia escolar, tutoría y educación sexual integral**, que son
las que responden al problema planteado.

In [6]:
# Variables de identificación
columnas_id = [
    "COD_MOD", "ANEXO", "CODLOCAL", "NROCED", "CEN_EDU", "DISTRITO",
    "LOCALIDAD", "CODGEO", "NIV_MOD", "GES_DEP",
]

# Variables de convivencia escolar y violencia (sección B de la cédula)
columnas_convivencia = [
    "P121B",        # cuenta con normas de convivencia escolar actualizadas
    "P122B",        # normas construidas con participación de estudiantes
    "P123B",        # cuenta con responsable de convivencia escolar
    "P124B",        # el colegio está afiliado al SíseVe
    "P124B_SI",     # si está afiliado: cuántos casos ha reportado
    "P124B_NO",     # si no está afiliado: razón de la no afiliación
    "P125B",        # cuenta con libro de registro de incidencias (Ley 29719)
    "P126B",        # en 2024 se hizo uso del libro de registro de incidencias
    "P126B_SI_1",   # si lo usó: cuántas incidencias se registraron en 2024
    "P127B",        # el director consulta el RNSSC para verificar al personal
    "P128B",        # el director conoce los protocolos de atención de violencia
    "P129B",        # sabe que los hechos deben reportarse también a la UGEL
]

# Variables de tutoría, ESI y acompañamiento
columnas_tutoria = [
    "P112A", "P113A", "P114A",
    "P116A",        # actividad con familias con eje en Educación Sexual Integral
    "P121A",        # criterio de distribución de estudiantes en secciones
    "P122A",        # acciones para atender interrupción de estudios
    "P103B", "P105B", "P106B",
]

columnas_solicitadas = columnas_id + columnas_convivencia + columnas_tutoria

columnas_existentes = [c for c in columnas_solicitadas if c in df_censo.columns]
faltantes = [c for c in columnas_solicitadas if c not in df_censo.columns]

df_censo_final = df_censo[columnas_existentes].copy()

print(f"Variables solicitadas: {len(columnas_solicitadas)}")
print(f"Variables encontradas: {len(columnas_existentes)}")
print(f"Variables no encontradas: {faltantes}")
display(df_censo_final.head())

Variables solicitadas: 31
Variables encontradas: 31
Variables no encontradas: []


,COD_MOD,ANEXO,CODLOCAL,NROCED,CEN_EDU,DISTRITO,LOCALIDAD,CODGEO,NIV_MOD,GES_DEP,P121B,P122B,P123B,P124B,P124B_SI,P124B_NO,P125B,P126B,P126B_SI_1,P127B,P128B,P129B,P112A,P113A,P114A,P116A,P121A,P122A,P103B,P105B,P106B
0,1720556,0,852373,3AP,SIR ALEXANDER FLEMING,CERCADO,,040101,B0,B4,SI,SI,SI,NO,,2,SI,NO,,SI,SI,SI,NO,NS,NO,,6,NO,SI,NO,
1,0313890,0,073181,3AP,40513 SAN MIGUEL ARCANGEL,ALCA,ANEXO DE CAHUANA,040802,B0,A1,SI,SI,SI,SI,0,,SI,NO,,NO,SI,SI,SI,SI,SI,SI,8,SI,SI,SI,SI
2,1495092,0,605028,3AP,33421,PILLCO MARCA,CAYHUAYNA,100111,B0,A1,SI,SI,SI,SI,0,,SI,SI,1,SI,SI,SI,SI,NO,SI,SI,9,SI,SI,SI,NO
3,1579861,0,651751,3AP,SANTANDER DE PUENTE PIEDRA,PUENTE PIEDRA,SANTA PAULA,150125,B0,B4,SI,NO,SI,SI,0,,SI,NO,,NO,NO,SI,NO,NO,SI,NO,6,NO,SI,SI,SI
4,1529767,0,553166,3AP,COLEGIO SUPERIOR DE CIENCIAS NEWTON VIANNEY,YURA,,040128,B0,B4,SI,NO,SI,SI,0,,SI,NO,,NO,NO,SI,NO,NO,NO,,1,NO,NO,,


In [7]:
print(f"Total de registros en df_censo_final: {len(df_censo_final)}")
print(f"Dimensiones (filas, columnas): {df_censo_final.shape}")
print("\nTipos de datos:")
display(df_censo_final.dtypes)

Total de registros en df_censo_final: 37354
Dimensiones (filas, columnas): (37354, 31)

Tipos de datos:


,0
COD_MOD,object
ANEXO,object
CODLOCAL,object
NROCED,object
CEN_EDU,object
DISTRITO,object
LOCALIDAD,object
CODGEO,object
NIV_MOD,object
GES_DEP,object


### <font color=skyblue> 1.3 Descripción de datos — Lineal_3AP_1 </font>

In [8]:
print("Resumen de df_censo_final:")
display(df_censo_final.describe(include="all").T.head(30))

Resumen de df_censo_final:


,count,unique,top,freq
COD_MOD,37354,37354,0403493,1
ANEXO,37354,1,0,37354
CODLOCAL,37354,37349,322232,2
NROCED,37354,1,3AP,37354
CEN_EDU,37354,33317,PRIMARIA,51
DISTRITO,37354,2899,SAN JUAN DE LURIGANCHO,453
LOCALIDAD,37354,20115,,1158
CODGEO,37354,1889,150132,490
NIV_MOD,37354,1,B0,37354
GES_DEP,37354,10,A1,29024


### <font color=skyblue> 1.4 Chequeo de faltantes y valores distintos — Lineal_3AP_1 </font>

En esta fuente los faltantes **no vienen como `NaN`**: vienen como **cadena vacía**,
porque el `.dbf` guarda todo como texto de ancho fijo. Por eso se cuentan los dos casos.

In [9]:
vacio_censo = df_censo_final.isna() | (df_censo_final.astype(str).apply(lambda s: s.str.strip()) == "")

resumen_censo = pd.DataFrame({
    "tipo_python": df_censo_final.dtypes.values,
    "n_faltantes": vacio_censo.sum().values,
    "porcentaje_faltantes": (vacio_censo.mean() * 100).round(2).values,
    "valores_distintos": df_censo_final.nunique().values,
}, index=df_censo_final.columns)

display(resumen_censo)

,tipo_python,n_faltantes,porcentaje_faltantes,valores_distintos
COD_MOD,object,0,0.00,37354
ANEXO,object,0,0.00,1
CODLOCAL,object,0,0.00,37349
NROCED,object,0,0.00,1
CEN_EDU,object,0,0.00,33317
DISTRITO,object,1,0.00,2899
LOCALIDAD,object,1158,3.10,20115
CODGEO,object,0,0.00,1889
NIV_MOD,object,0,0.00,1
GES_DEP,object,0,0.00,10


In [10]:
# Distribución de las variables clave de convivencia
for col in ["P124B", "P125B", "P126B", "P123B", "P128B", "P116A"]:
    print(f"--- {col} ---")
    print(df_censo_final[col].value_counts(dropna=False).to_dict())
    print()

--- P124B ---
{'SI': 30709, 'NO': 6611, '': 34}

--- P125B ---
{'SI': 31045, 'NO': 6269, '': 40}

--- P126B ---
{'NO': 27071, '': 5875, 'SI': 4408}

--- P123B ---
{'SI': 32531, 'NO': 4800, '': 23}

--- P128B ---
{'SI': 32517, 'NO': 4812, '': 25}

--- P116A ---
{'SI': 16482, 'NO': 13158, '': 7714}



### <font color=skyblue> 1.5 Chequeo de duplicados — Lineal_3AP_1 </font>

In [11]:
df_censo_final = normalizar_llave(df_censo_final)

print("Filas duplicadas exactas:", df_censo_final.duplicated().sum())
print("IDs repetidos (COD_MOD + ANEXO):", df_censo_final.duplicated(subset=LLAVE).sum())

Filas duplicadas exactas: 0
IDs repetidos (COD_MOD + ANEXO): 0


### <font color=skyblue> 1.6 Conversión de tipos y estandarización de texto — Lineal_3AP_1 </font>

In [12]:
# Variables SI/NO: quitar espacios y unificar mayúsculas. El vacío pasa a NaN
# porque "no respondió" no es lo mismo que "NO".
columnas_si_no = ["P121B", "P122B", "P123B", "P124B", "P125B", "P126B",
                  "P127B", "P128B", "P129B", "P116A", "P122A",
                  "P112A", "P113A", "P114A", "P103B", "P105B", "P106B"]

for col in columnas_si_no:
    df_censo_final[col] = (df_censo_final[col].astype(str).str.strip().str.upper()
                           .replace({"": np.nan, "NAN": np.nan, "NONE": np.nan}))

# Variables de texto descriptivo
for col in ["DISTRITO", "LOCALIDAD", "CEN_EDU"]:
    df_censo_final[col] = df_censo_final[col].astype(str).str.strip().str.title()

# Variables de conteo
df_censo_final["P124B_SI"] = pd.to_numeric(df_censo_final["P124B_SI"], errors="coerce")
df_censo_final["P126B_SI_1"] = pd.to_numeric(df_censo_final["P126B_SI_1"], errors="coerce")

print("Estandarización finalizada.")
display(df_censo_final[["P124B", "P124B_SI", "P126B", "P126B_SI_1"]].head(10))

Estandarización finalizada.


,P124B,P124B_SI,P126B,P126B_SI_1
0,NO,NaN,NO,NaN
1,SI,0.0,NO,NaN
2,SI,0.0,SI,1.0
3,SI,0.0,NO,NaN
4,SI,0.0,NO,NaN
5,SI,0.0,NO,NaN
6,NO,NaN,NaN,NaN
7,SI,1.0,SI,4.0
8,SI,0.0,NO,NaN
9,SI,0.0,NO,NaN


### <font color=skyblue> 1.7 Construcción de las variables de violencia escolar — Lineal_3AP_1 </font>

Estas dos variables son el centro del trabajo, y su tratamiento requiere una decisión explícita.

`P124B_SI` y `P126B_SI_1` son **preguntas condicionadas**: solo se llenan si la respuesta
a la pregunta filtro (`P124B`, `P126B`) fue *Sí*. Por eso un vacío puede significar tres
cosas distintas:

| Respuesta filtro | Qué significa el vacío | Cómo se trata |
|---|---|---|
| `SI` | El colegio sí registró, pero no anotó el número | Se imputa **0** (son 2 y 4 casos) |
| `NO` | El colegio no está afiliado / no usó el libro | Es un **0 estructural**, no un faltante |
| vacío | El director no respondió la pregunta filtro | Se deja como **NaN**: no se puede inventar |

Esto corrige el tratamiento de la versión anterior, donde estos vacíos se imputaban con la
**mediana general**. Imputar la mediana asigna incidencias de violencia a colegios que
declararon explícitamente no haber registrado ninguna, lo que infla artificialmente el indicador.

In [13]:
# Casos reportados al SíseVe
df_censo_final["casos_siseve"] = np.where(
    df_censo_final["P124B"] == "SI", df_censo_final["P124B_SI"].fillna(0),
    np.where(df_censo_final["P124B"] == "NO", 0, np.nan))

# Incidencias registradas en el libro de incidencias durante 2024
df_censo_final["incidencias_libro"] = np.where(
    df_censo_final["P126B"] == "SI", df_censo_final["P126B_SI_1"].fillna(0),
    np.where(df_censo_final["P126B"] == "NO", 0, np.nan))

print("casos_siseve      -> con dato:", df_censo_final["casos_siseve"].notna().sum(),
      "| sin dato:", df_censo_final["casos_siseve"].isna().sum())
print("incidencias_libro -> con dato:", df_censo_final["incidencias_libro"].notna().sum(),
      "| sin dato:", df_censo_final["incidencias_libro"].isna().sum())

display(df_censo_final[["casos_siseve", "incidencias_libro"]].describe().round(2))

casos_siseve      -> con dato: 37320 | sin dato: 34
incidencias_libro -> con dato: 31479 | sin dato: 5875


,casos_siseve,incidencias_libro
count,37320.00,31479.00
mean,0.18,0.59
std,1.15,7.07
min,0.00,0.00
25%,0.00,0.00
50%,0.00,0.00
75%,0.00,0.00
max,99.00,1000.00


### <font color=skyblue> 1.8 Detección de valores fuera de rango — Lineal_3AP_1 </font>

In [14]:
# El año escolar peruano tiene alrededor de 190 días lectivos. Un colegio de primaria
# que declare más de 365 incidencias registraría más de una por día del año: es
# implausible y probablemente un error de digitación (p. ej. anotar el código en lugar
# de la cantidad).
UMBRAL = 365

for col in ["casos_siseve", "incidencias_libro"]:
    fuera = df_censo_final[df_censo_final[col] > UMBRAL]
    print(f"{col}: {len(fuera)} registros por encima de {UMBRAL} (máximo observado: {df_censo_final[col].max():.0f})")
    if len(fuera):
        display(fuera[["COD_MOD", "CEN_EDU", "DISTRITO", col]])

casos_siseve: 0 registros por encima de 365 (máximo observado: 99)
incidencias_libro: 2 registros por encima de 365 (máximo observado: 1000)


,COD_MOD,CEN_EDU,DISTRITO,incidencias_libro
18528,1728054,Futura Schools,Veintiseis De Octubre,387.0
34915,0354423,Salesiano Don Bosco,Castilla,1000.0


In [15]:
# Se convierten a NaN en lugar de imputarse: son pocos casos y no hay una regla
# de negocio que permita corregir el valor real.
for col in ["casos_siseve", "incidencias_libro"]:
    df_censo_final.loc[df_censo_final[col] > UMBRAL, col] = np.nan

print("Valores máximos después del tratamiento:")
display(df_censo_final[["casos_siseve", "incidencias_libro"]].max())

Valores máximos después del tratamiento:


,0
casos_siseve,99.0
incidencias_libro,230.0


**¿Por qué no se usó la regla del IQR?** Es el criterio estándar para detectar atípicos, pero aquí no
sirve: más del 85% de los colegios registra cero, así que Q1 = Q3 = 0, el IQR vale 0 y el límite superior
también. La regla marcaría como "atípico" a **cualquier colegio que registre algo**: marcaría la señal en
lugar del error. Por eso se usó una regla de negocio (365), que sí distingue un valor imposible de un
valor simplemente alto.

In [16]:
def limite_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    return q3 + 1.5 * (q3 - q1)

for col in ["casos_siseve", "incidencias_libro"]:
    limite = limite_iqr(df_censo_final[col])
    marcados = (df_censo_final[col] > limite).sum()
    con_dato = df_censo_final[col].notna().sum()
    print(f"{col}: límite IQR = {limite:.0f} -> marcaría {marcados} colegios "
          f"({marcados / con_dato:.1%} de los que tienen dato)")

casos_siseve: límite IQR = 0 -> marcaría 3280 colegios (8.8% de los que tienen dato)
incidencias_libro: límite IQR = 0 -> marcaría 4391 colegios (13.9% de los que tienen dato)


### <font color=skyblue> 1.9 Decisiones de limpieza justificadas — Lineal_3AP_1 </font>

In [17]:
decisiones_censo = pd.DataFrame([
    ["Nombres de columnas", "Mezcla de mayúsculas y minúsculas",
     "Convertir todo a mayúsculas",
     "Evita errores de referencia al cruzar con otras fuentes."],
    ["Variables SI/NO", "El .dbf guarda los faltantes como cadena vacía, no como NaN",
     "Convertir la cadena vacía a NaN",
     "'No respondió' no es lo mismo que 'NO'; mezclarlos sesga los porcentajes."],
    ["P124B_SI / P126B_SI_1", "Preguntas condicionadas: el vacío tiene tres significados distintos",
     "0 si la pregunta filtro fue NO; 0 si fue SI sin número; NaN si no respondió el filtro",
     "Imputar la mediana (versión anterior) asigna violencia a colegios que declararon no haber registrado ninguna."],
    ["casos_siseve / incidencias_libro", "Valores implausibles por encima de 365 al año",
     "Convertir a NaN, sin imputar",
     "Supera el número de días lectivos del año escolar; no hay regla para corregir el valor real."],
    ["incidencias_libro (regla IQR)", "Más del 85% de los colegios registra 0, por lo que el IQR es 0",
     "No usar el IQR; usar la regla de negocio (365)",
     "Con IQR = 0 se marcaría como atípico a todo colegio que registra algo: sería marcar la señal."],
    ["DISTRITO, LOCALIDAD, CEN_EDU", "Espacios extra y formato inconsistente",
     "strip() y Title Case",
     "Permite agrupar sin que el mismo distrito aparezca dos veces."],
], columns=["variable", "problema", "decisión", "justificación"])

display(decisiones_censo)

,variable,problema,decisión,justificación
0,Nombres de columnas,Mezcla de mayúsculas y minúsculas,Convertir todo a mayúsculas,Evita errores de referencia al cruzar con otra...
1,Variables SI/NO,El .dbf guarda los faltantes como cadena vacía...,Convertir la cadena vacía a NaN,'No respondió' no es lo mismo que 'NO'; mezcla...
2,P124B_SI / P126B_SI_1,Preguntas condicionadas: el vacío tiene tres s...,0 si la pregunta filtro fue NO; 0 si fue SI si...,Imputar la mediana (versión anterior) asigna v...
3,casos_siseve / incidencias_libro,Valores implausibles por encima de 365 al año,"Convertir a NaN, sin imputar",Supera el número de días lectivos del año esco...
4,incidencias_libro (regla IQR),"Más del 85% de los colegios registra 0, por lo...",No usar el IQR; usar la regla de negocio (365),Con IQR = 0 se marcaría como atípico a todo co...
5,"DISTRITO, LOCALIDAD, CEN_EDU",Espacios extra y formato inconsistente,strip() y Title Case,Permite agrupar sin que el mismo distrito apar...


## <font color=yellow> 2. Tratamiento de datos — Resultado del Ejercicio Educativo 2025 </font>

### <font color=skyblue> 2.1 Importación de datos — Resultado del Ejercicio Educativo </font>

Esta fuente proviene del archivo `Resultado_2025.dbf` (ESCALE), cuya información se
construye a partir del **SIAGIE**. Es un registro administrativo, no una denuncia
voluntaria, por lo que no sufre el sesgo de subregistro por falta de canales de reporte.

**Transformación previa a la carga.** El `.dbf` original viene en **formato largo**: cada
fila es una combinación de colegio × cuadro × situación, y las columnas `D01` a `D12`
son grado × sexo. Para llevarlo a la unidad de análisis del trabajo (un colegio = una fila)
se filtró la cédula `3BP` (Primaria) y se pivoteó. El código de esa preparación fue:

```python
res = pd.read_csv("res3bp.csv", dtype=str)     # Resultado_2025.dbf filtrado a NROCED == "3BP"
D = [f"D{i:02d}" for i in range(1, 13)]        # 1er a 6to grado x (Hombre, Mujer)
res["TOTAL"] = res[D].astype(int).sum(axis=1)

# C101 = situación final del estudiante, C102 = motivo del retiro, C201 = recuperación
c101 = res[res["CUADRO"] == "C101"].pivot_table(
    index=["COD_MOD", "ANEXO"], columns="TIPDATO", values="TOTAL", aggfunc="sum", fill_value=0)
```

El resultado de esa transformación es el archivo `resultado_primaria_2025.parquet`.

In [18]:
df_resultado = leer_de_hugging_face("resultado_primaria_2025.parquet")
df_resultado = normalizar_llave(df_resultado)

print("Resultado del Ejercicio Educativo (Primaria):", df_resultado.shape)
display(df_resultado.head())

Resultado del Ejercicio Educativo (Primaria): (38036, 41)


,COD_MOD,ANEXO,NIV_MOD,GES_DEP,AREA_CENSO,CODGEO,CODOOII,FUENTE,res_promovido,res_requiere_recup,res_permanece_grado,res_posterga_eval,res_retirado,res_fallecido,res_promovido_h,res_requiere_recup_h,res_permanece_grado_h,res_posterga_eval_h,res_retirado_h,res_fallecido_h,res_promovido_m,res_requiere_recup_m,res_permanece_grado_m,res_posterga_eval_m,res_retirado_m,res_fallecido_m,motret_economica,motret_violencia,motret_enfermedad,motret_trabajo_infantil,motret_labores_agricolas,motret_adiccion,motret_otro,recup_promovido,recup_permanece_grado,recup_no_se_present,res_evaluados,tasa_repeticion,tasa_retiro,tasa_recup_ped,motret_total
0,0200014,0,B0,A1,2,030701,030007,SIAGIE,7,0,0,0,0,0,2,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,0.0,0.0,0.0,0
1,0200022,0,B0,A1,2,030701,030007,SIAGIE,5,0,0,0,0,0,4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0.0,0.0,0.0,0
2,0200030,0,B0,A1,2,030701,030007,SIAGIE,10,0,0,0,0,0,2,0,0,0,0,0,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10,0.0,0.0,0.0,0
3,0200048,0,B0,A1,2,030701,030007,SIAGIE,2,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0.0,0.0,0.0,0
4,0200055,0,B0,A1,2,030701,030007,SIAGIE,4,0,0,0,0,0,1,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0.0,0.0,0.0,0


In [19]:
print("Columnas disponibles:")
for c in df_resultado.columns:
    print(" -", c)

Columnas disponibles:
 - COD_MOD
 - ANEXO
 - NIV_MOD
 - GES_DEP
 - AREA_CENSO
 - CODGEO
 - CODOOII
 - FUENTE
 - res_promovido
 - res_requiere_recup
 - res_permanece_grado
 - res_posterga_eval
 - res_retirado
 - res_fallecido
 - res_promovido_h
 - res_requiere_recup_h
 - res_permanece_grado_h
 - res_posterga_eval_h
 - res_retirado_h
 - res_fallecido_h
 - res_promovido_m
 - res_requiere_recup_m
 - res_permanece_grado_m
 - res_posterga_eval_m
 - res_retirado_m
 - res_fallecido_m
 - motret_economica
 - motret_violencia
 - motret_enfermedad
 - motret_trabajo_infantil
 - motret_labores_agricolas
 - motret_adiccion
 - motret_otro
 - recup_promovido
 - recup_permanece_grado
 - recup_no_se_present
 - res_evaluados
 - tasa_repeticion
 - tasa_retiro
 - tasa_recup_ped
 - motret_total


### <font color=skyblue> 2.2 Chequeo de faltantes — Resultado del Ejercicio Educativo </font>

In [20]:
faltantes_resultado = pd.DataFrame({
    "n_faltantes": df_resultado.isna().sum(),
    "porcentaje": (df_resultado.isna().mean() * 100).round(2),
})
display(faltantes_resultado[faltantes_resultado["n_faltantes"] > 0])

print("Variables sin ningún faltante:",
      (faltantes_resultado["n_faltantes"] == 0).sum(), "de", len(faltantes_resultado))

,n_faltantes,porcentaje


Variables sin ningún faltante: 41 de 41


### <font color=skyblue> 2.3 Chequeo de duplicados — Resultado del Ejercicio Educativo </font>

In [21]:
print("Filas duplicadas exactas:", df_resultado.duplicated().sum())
print("IDs repetidos (COD_MOD + ANEXO):", df_resultado.duplicated(subset=LLAVE).sum())

# A diferencia de la base de denuncias, esta fuente sí tiene identificador de colegio,
# por lo que la unicidad de la llave se puede verificar directamente.

Filas duplicadas exactas: 0
IDs repetidos (COD_MOD + ANEXO): 0


### <font color=skyblue> 2.4 Tipos de datos — Resultado del Ejercicio Educativo </font>

In [22]:
display(df_resultado.dtypes)

,0
COD_MOD,object
ANEXO,object
NIV_MOD,object
GES_DEP,object
AREA_CENSO,object
CODGEO,object
CODOOII,object
FUENTE,object
res_promovido,int32
res_requiere_recup,int32


### <font color=skyblue> 2.5 Valores fuera de rango — Resultado del Ejercicio Educativo </font>

In [23]:
# 1) La suma de las situaciones debe coincidir con el total de evaluados
situaciones = ["res_promovido", "res_requiere_recup", "res_permanece_grado",
               "res_posterga_eval", "res_retirado", "res_fallecido"]
descuadre = (df_resultado[situaciones].sum(axis=1) != df_resultado["res_evaluados"])
print("Colegios donde la suma no cuadra con res_evaluados:", descuadre.sum())

# 2) Colegios sin ningún estudiante evaluado (división por cero al calcular tasas)
print("Colegios con res_evaluados = 0:", (df_resultado["res_evaluados"] == 0).sum())

# 3) Tasas fuera del intervalo [0, 1]
for col in ["tasa_repeticion", "tasa_retiro", "tasa_recup_ped"]:
    fuera = ((df_resultado[col] < 0) | (df_resultado[col] > 1)).sum()
    print(f"{col}: {fuera} valores fuera de [0, 1]")

# 4) Coherencia entre el total de retirados y el desglose por motivo
motivos = [c for c in df_resultado.columns if c.startswith("motret_") and c != "motret_total"]
sin_motivo = (df_resultado["res_retirado"] - df_resultado["motret_total"])
print("\nRetirados sin motivo declarado (total):", int(sin_motivo.clip(lower=0).sum()))

Colegios donde la suma no cuadra con res_evaluados: 0
Colegios con res_evaluados = 0: 0
tasa_repeticion: 0 valores fuera de [0, 1]
tasa_retiro: 0 valores fuera de [0, 1]
tasa_recup_ped: 0 valores fuera de [0, 1]

Retirados sin motivo declarado (total): 0


In [24]:
display(df_resultado[["res_evaluados", "res_promovido", "res_permanece_grado",
                      "res_retirado", "tasa_repeticion", "tasa_retiro"]].describe().round(4))

,res_evaluados,res_promovido,res_permanece_grado,res_retirado,tasa_repeticion,tasa_retiro
count,38036.0000,38036.0000,38036.0000,38036.0000,38036.0000,38036.0000
mean,93.8640,89.6163,1.1689,0.3313,0.0148,0.0037
std,165.1024,157.2782,3.9890,1.6125,0.0468,0.0250
min,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,12.0000,11.0000,0.0000,0.0000,0.0000,0.0000
50%,31.0000,30.0000,0.0000,0.0000,0.0000,0.0000
75%,94.0000,91.0000,0.0000,0.0000,0.0000,0.0000
max,2234.0000,2177.0000,137.0000,48.0000,1.0000,1.0000


### <font color=skyblue> 2.6 Motivos de retiro — Resultado del Ejercicio Educativo </font>

In [25]:
resumen_motivos = pd.DataFrame({
    "total_estudiantes": df_resultado[motivos].sum().astype(int),
    "colegios_con_al_menos_un_caso": (df_resultado[motivos] > 0).sum(),
}).sort_values("total_estudiantes", ascending=False)

display(resumen_motivos)

,total_estudiantes,colegios_con_al_menos_un_caso
motret_otro,10898,3584
motret_enfermedad,771,484
motret_economica,766,362
motret_labores_agricolas,142,67
motret_trabajo_infantil,14,12
motret_violencia,6,5
motret_adiccion,5,5


**Observación importante para el análisis.**
El motivo *Violencia* aparece en apenas un puñado de colegios de todo el país. Por eso
**no se usa como variable dependiente**: con esa frecuencia cualquier modelo o
comparación sería ruido. Se conserva como variable descriptiva y el análisis de violencia
se apoya en `casos_siseve` e `incidencias_libro` (sección 1), que sí tienen cobertura nacional.

Las variables de resultado que sí se usan son la **repetición** (`res_permanece_grado`) y
el **retiro** (`res_retirado`), que están bien pobladas.

### <font color=skyblue> 2.7 Decisiones de limpieza justificadas — Resultado del Ejercicio Educativo </font>

In [26]:
decisiones_resultado = pd.DataFrame([
    ["Estructura de la tabla", "El .dbf viene en formato largo (colegio x cuadro x situación)",
     "Pivotear a una fila por colegio antes de publicar el parquet",
     "Es la única forma de que la unidad de análisis coincida con la del censo y el padrón."],
    ["Cédulas", "El archivo contiene todos los niveles educativos (1B, 2B, 3BP, 3BS, ...)",
     "Filtrar NROCED == '3BP'",
     "El problema planteado se refiere exclusivamente a educación primaria."],
    ["res_evaluados = 0", "Genera división por cero al calcular tasas",
     "Dejar la tasa como NaN en esos casos",
     "Imputar 0 afirmaría que no hubo repetición, cuando en realidad no hubo estudiantes evaluados."],
    ["motret_violencia", "Solo unos pocos colegios en todo el país declaran este motivo",
     "Conservarla como variable descriptiva, no como variable dependiente",
     "Una variable casi constante no permite comparar ni modelar; usarla llevaría a conclusiones espurias."],
], columns=["variable", "problema", "decisión", "justificación"])

display(decisiones_resultado)

,variable,problema,decisión,justificación
0,Estructura de la tabla,El .dbf viene en formato largo (colegio x cuad...,Pivotear a una fila por colegio antes de publi...,Es la única forma de que la unidad de análisis...
1,Cédulas,El archivo contiene todos los niveles educativ...,Filtrar NROCED == '3BP',El problema planteado se refiere exclusivament...
2,res_evaluados = 0,Genera división por cero al calcular tasas,Dejar la tasa como NaN en esos casos,"Imputar 0 afirmaría que no hubo repetición, cu..."
3,motret_violencia,Solo unos pocos colegios en todo el país decla...,"Conservarla como variable descriptiva, no como...",Una variable casi constante no permite compara...


## <font color=yellow> 3. Tratamiento de datos — Matrícula 2025 </font>

### <font color=skyblue> 3.1 Importación de datos — Matrícula </font>

La matrícula cumple un papel específico: es el **denominador**. Sin ella, 5 incidencias en
un colegio de 30 estudiantes y 5 en uno de 800 se ven iguales, cuando representan
realidades opuestas.

In [27]:
df_matricula = leer_de_hugging_face("matricula_primaria_2025.parquet")
df_matricula = normalizar_llave(df_matricula)

print("Matrícula (Primaria):", df_matricula.shape)
display(df_matricula.head())

Matrícula (Primaria): (38338, 32)


,COD_MOD,ANEXO,NIV_MOD,GES_DEP,AREA_CENSO,CODGEO,CODOOII,CODLOCAL,IMPUTADO,mat_g1_h,mat_g1_m,mat_g2_h,mat_g2_m,mat_g3_h,mat_g3_m,mat_g4_h,mat_g4_m,mat_g5_h,mat_g5_m,mat_g6_h,mat_g6_m,mat_total,mat_hombres,mat_mujeres,mat_g1,mat_g2,mat_g3,mat_g4,mat_g5,mat_g6,n_turnos,pct_mujeres
0,0200014,0,B0,A1,2,030701,030007,053828,1_INFORMANTE,0,1,1,0,1,1,0,2,0,0,0,1,7,2,5,1,1,2,2,0,1,1,0.7143
1,0200022,0,B0,A1,2,030701,030007,053833,1_INFORMANTE,1,0,0,0,0,0,1,0,2,0,0,1,5,4,1,1,0,0,1,2,1,1,0.2000
2,0200030,0,B0,A1,2,030701,030007,053847,1_INFORMANTE,1,1,0,3,1,0,0,1,0,1,0,2,10,2,8,2,3,1,1,1,2,1,0.8000
3,0200048,0,B0,A1,2,030701,030007,053852,1_INFORMANTE,2,0,0,0,0,0,0,0,0,0,0,0,2,2,0,2,0,0,0,0,0,1,0.0000
4,0200055,0,B0,A1,2,030701,030007,053866,1_INFORMANTE,0,0,1,1,0,0,0,2,0,0,0,0,4,1,3,0,2,0,2,0,0,1,0.7500


### <font color=skyblue> 3.2 Chequeos de calidad — Matrícula </font>

In [28]:
print("Filas duplicadas exactas:", df_matricula.duplicated().sum())
print("IDs repetidos (COD_MOD + ANEXO):", df_matricula.duplicated(subset=LLAVE).sum())
print("Colegios con matrícula total = 0:", (df_matricula["mat_total"] == 0).sum())
print("Colegios con matrícula negativa:", (df_matricula["mat_total"] < 0).sum())

# La suma de los seis grados debe dar el total
grados = [f"mat_g{i}" for i in range(1, 7)]
print("Descuadres entre grados y total:",
      (df_matricula[grados].sum(axis=1) != df_matricula["mat_total"]).sum())

display(df_matricula[["mat_total", "mat_hombres", "mat_mujeres", "pct_mujeres",
                      "n_turnos"]].describe().round(2))

Filas duplicadas exactas: 0
IDs repetidos (COD_MOD + ANEXO): 0
Colegios con matrícula total = 0: 0
Colegios con matrícula negativa: 0
Descuadres entre grados y total: 0


,mat_total,mat_hombres,mat_mujeres,pct_mujeres,n_turnos
count,38338.00,38338.00,38338.00,38338.00,38338.00
mean,95.64,48.57,47.08,0.48,1.04
std,169.71,87.85,86.94,0.15,0.20
min,1.00,0.00,0.00,0.00,1.00
25%,12.00,6.00,6.00,0.42,1.00
50%,32.00,16.00,15.00,0.49,1.00
75%,94.75,48.00,45.00,0.54,1.00
max,2234.00,1628.00,1425.00,1.00,3.00


In [29]:
# La variable IMPUTADO indica cómo se obtuvo el dato de matrícula en el censo
print("Origen del dato de matrícula:")
display(df_matricula["IMPUTADO"].value_counts(dropna=False))

Origen del dato de matrícula:


,count
IMPUTADO,
1_INFORMANTE,37061
3_IMPUTADO_TOTAL,1189
2_IMPUTADO_PARCIAL,88


### <font color=skyblue> 3.3 Valores atípicos — Matrícula </font>

Aquí sí funciona la regla del IQR, porque la matrícula no se concentra en cero. Pero **detectar no
significa eliminar**: hay que mirar qué son los valores marcados antes de decidir.

In [30]:
limite = limite_iqr(df_matricula["mat_total"])
atipicos = df_matricula[df_matricula["mat_total"] > limite]

print(f"Límite IQR: {limite:.0f} estudiantes -> {len(atipicos)} colegios ({len(atipicos) / len(df_matricula):.1%})")
print("Matrícula máxima:", df_matricula["mat_total"].max())
display(atipicos["mat_total"].describe().round(0))

Límite IQR: 219 estudiantes -> 4508 colegios (11.8%)
Matrícula máxima: 2234


,mat_total
count,4508.0
mean,479.0
std,247.0
min,219.0
25%,296.0
50%,400.0
75%,591.0
max,2234.0


Son colegios grandes reales (la matrícula cuadra con la suma de grados y no hay valores imposibles),
no errores de captura. **Se conservan.** Su tamaño se controla más adelante con la variable
`tam_colegio`.

### <font color=skyblue> 3.4 Decisiones de limpieza justificadas — Matrícula </font>

In [31]:
# Un colegio con matrícula 0 no puede servir de denominador
df_matricula["mat_total"] = df_matricula["mat_total"].replace(0, np.nan)

decisiones_matricula = pd.DataFrame([
    ["Cuadro C201", "La tabla trae una fila por turno de enseñanza",
     "Sumar los turnos por colegio antes de publicar el parquet",
     "Un colegio con turno mañana y tarde debe contar como un solo colegio."],
    ["mat_total = 0", "Se usará como denominador de las tasas",
     "Convertir a NaN",
     "Evita divisiones por cero y tasas infinitas."],
    ["mat_total alta (IQR)", "El IQR marca como atípico a ~12% de los colegios",
     "Conservar y controlar por tamaño (tam_colegio)",
     "Son colegios grandes reales, no errores; eliminarlos sesgaría la muestra hacia colegios pequeños."],
    ["IMPUTADO", "Parte de la matrícula fue imputada por el MINEDU, no declarada",
     "Conservar la variable en la base final",
     "Permite documentar la limitación y, de ser necesario, excluir esos casos."],
], columns=["variable", "problema", "decisión", "justificación"])

display(decisiones_matricula)

,variable,problema,decisión,justificación
0,Cuadro C201,La tabla trae una fila por turno de enseñanza,Sumar los turnos por colegio antes de publicar...,Un colegio con turno mañana y tarde debe conta...
1,mat_total = 0,Se usará como denominador de las tasas,Convertir a NaN,Evita divisiones por cero y tasas infinitas.
2,mat_total alta (IQR),El IQR marca como atípico a ~12% de los colegios,Conservar y controlar por tamaño (tam_colegio),"Son colegios grandes reales, no errores; elimi..."
3,IMPUTADO,Parte de la matrícula fue imputada por el MINE...,Conservar la variable en la base final,"Permite documentar la limitación y, de ser nec..."


## <font color=yellow> 4. Tratamiento de datos — Padron_web </font>

### <font color=skyblue> 4.1 Importación de datos — Padron_web </font>

In [32]:
df_padron = leer_de_hugging_face("padron_iiee.parquet")
print("Padrón de IIEE:", df_padron.shape)
display(df_padron.head(5))

"""
# Lectura original desde el archivo .dbf:
from dbfread import DBF
tabla_padron = DBF("Padron_web.dbf", encoding="cp850", char_decode_errors="replace")
df_padron = pd.DataFrame(iter(tabla_padron))
"""

Padrón de IIEE: (180792, 45)


,CODINST,COD_MOD,ANEXO,CODLOCAL,CEN_EDU,NIV_MOD,D_NIV_MOD,D_FORMA,TIPSSEXO,D_TIPSSEXO,GESTION,D_GESTION,GES_DEP,D_GES_DEP,DIRECTOR,TELEFONO,EMAIL,PAGWEB,DIR_CEN,LOCALIDAD,CODCP_INEI,CODCCPP,CEN_POB,AREA_CENSO,DAREACENSO,CODGEO,D_DPTO,D_PROV,D_DIST,D_REGION,CODOOII,D_DREUGEL,NLAT_IE,NLONG_IE,TIPOPROG,D_TIPOPROG,COD_TUR,D_COD_TUR,NRORUC,RZSOCIAL,PROMOTOR,ESTADO,D_ESTADO,FECHAREG,FECHA_ACT
0,24953981,0415547,0,016100,123,A2,Inicial - Jardín,Escolarizada,3,Mixto,1,Pública de gestión directa,A1,Sector Educación,CACERES DE MAUTINO YONNY ESCOLASTICA,,,,JIRON TERESA GONZALES DE FANNY 543,NICRUPAMPA,0201050001,129688,CENTENARIO,1,Urbana,020105,ANCASH,HUARAZ,INDEPENDENCIA,DRE ANCASH,020001,UGEL HUARAZ,-9.518850,-77.531910,,,11,Mañana,,,,1,Activo,,05-08-2026
1,25273230,0415638,0,015172,122,A2,Inicial - Jardín,Escolarizada,3,Mixto,1,Pública de gestión directa,A1,Sector Educación,CACERES DE MAUTINO YONNY ESCOLASTICA,043-420252,,,JIRON 28 DE JULIO S/N,HUARUPAMPA,0201010001,114517,HUARUPAMPA,1,Urbana,020101,ANCASH,HUARAZ,HUARAZ,DRE ANCASH,020001,UGEL HUARAZ,-9.530670,-77.531960,,,11,Mañana,,,,1,Activo,02-05-1942,05-08-2026
2,20414731,0415646,0,015186,233,A2,Inicial - Jardín,Escolarizada,3,Mixto,1,Pública de gestión directa,A1,Sector Educación,LOPEZ MONGE MARIA ESTHER MONICA,043-427649 - 943940593,,,JIRON AMADEO FIGUEROA S/N,,,610619,LA SOLEDAD,1,Urbana,020101,ANCASH,HUARAZ,HUARAZ,DRE ANCASH,020001,UGEL HUARAZ,-9.531100,-77.522700,,,13,Mañana-Tarde,,,,1,Activo,04-08-1974,05-08-2026
3,26281962,0415877,0,016751,COLEGIO PARROQUIAL NUESTRA SEÑORA DEL SAGRADO ...,A2,Inicial - Jardín,Escolarizada,3,Mixto,3,Privada,B2,Comunidad o asociación religiosa,DAVILA CALVO ROMMEL SANTIAGO,043-421652,nsscj2005@yahoo.com,,JIRON LOS CAPULIES 163,,0201050001,129688,CENTENARIO,1,Urbana,020105,ANCASH,HUARAZ,INDEPENDENCIA,DRE ANCASH,020001,UGEL HUARAZ,-9.515578,-77.532393,,,13,Mañana-Tarde,20146930418,C.E.P. NUESTRA SEÑORA DEL SAGRADO CORAZON DE J...,,1,Activo,07-05-1962,05-08-2026
4,25111788,0567206,0,016119,268,A2,Inicial - Jardín,Escolarizada,3,Mixto,1,Pública de gestión directa,A1,Sector Educación,HUARAC CHAUCA REYNA AURORA,,,,MARIAN,,0201050064,124291,MARIAN,2,Rural,020105,ANCASH,HUARAZ,INDEPENDENCIA,DRE ANCASH,020001,UGEL HUARAZ,-9.513940,-77.504026,,,11,Mañana,,,,1,Activo,24-04-1981,05-08-2026


'\n# Lectura original desde el archivo .dbf:\nfrom dbfread import DBF\ntabla_padron = DBF("Padron_web.dbf", encoding="cp850", char_decode_errors="replace")\ndf_padron = pd.DataFrame(iter(tabla_padron))\n'

### <font color=skyblue> 4.2 Conversión de tipos de datos — Padron_web </font>

In [33]:
padron = normalizar_llave(df_padron)

padron["NLAT_IE"] = pd.to_numeric(padron["NLAT_IE"], errors="coerce")
padron["NLONG_IE"] = pd.to_numeric(padron["NLONG_IE"], errors="coerce")

padron["FECHAREG"] = pd.to_datetime(padron["FECHAREG"], errors="coerce", dayfirst=True)
padron["FECHA_ACT"] = pd.to_datetime(padron["FECHA_ACT"], errors="coerce", dayfirst=True)

padron["CODLOCAL"] = padron["CODLOCAL"].astype(str).str.strip().str.zfill(6)
padron["CODGEO"] = padron["CODGEO"].astype(str).str.strip().str.zfill(6)
padron["CODOOII"] = padron["CODOOII"].astype(str).str.strip().str.zfill(6)

# Coordenadas 0,0 significan "sin georreferenciar", no una ubicación real
sin_coord = (padron["NLAT_IE"] == 0) & (padron["NLONG_IE"] == 0)
padron.loc[sin_coord, ["NLAT_IE", "NLONG_IE"]] = np.nan

print("Tipos después de convertir:")
display(padron.dtypes.head(20))

Tipos después de convertir:


,0
CODINST,object
COD_MOD,object
ANEXO,object
CODLOCAL,object
CEN_EDU,object
NIV_MOD,object
D_NIV_MOD,object
D_FORMA,object
TIPSSEXO,object
D_TIPSSEXO,object


### <font color=skyblue> 4.3 Estandarización de nombres y categorías — Padron_web </font>

In [34]:
for col in ["D_NIV_MOD", "D_FORMA", "D_TIPSSEXO", "D_GESTION", "D_GES_DEP",
            "DAREACENSO", "D_ESTADO", "D_COD_TUR"]:
    padron[col] = padron[col].astype(str).str.strip()

for col in ["D_DPTO", "D_PROV", "D_DIST", "D_REGION", "D_DREUGEL"]:
    padron[col] = padron[col].astype(str).str.strip().str.upper()

# Quitar tildes para que no se dupliquen categorías
for col in ["D_REGION", "D_DREUGEL"]:
    padron[col] = (padron[col].str.normalize("NFKD")
                   .str.encode("ascii", errors="ignore").str.decode("utf-8"))

# Unificar GRE y DRE: son la misma entidad con distinta denominación regional
padron["D_REGION"] = padron["D_REGION"].replace({
    "GRE AREQUIPA": "DRE AREQUIPA",
    "GRE LA LIBERTAD": "DRE LA LIBERTAD",
    "GRE LAMBAYEQUE": "DRE LAMBAYEQUE",
})

print("Departamentos distintos:", padron["D_DPTO"].nunique())
print("Tipos de gestión:", padron["D_GESTION"].unique().tolist())

Departamentos distintos: 25
Tipos de gestión: ['Pública de gestión directa', 'Privada', 'Pública de gestión privada']


### <font color=skyblue> 4.4 Valores fuera de rango — Padron_web </font>

In [35]:
HOY = pd.Timestamp("today").normalize()

fuera_peru = padron[(padron["NLAT_IE"] < -18.4) | (padron["NLAT_IE"] > 0) |
                    (padron["NLONG_IE"] < -81.4) | (padron["NLONG_IE"] > -68.6)]
print("Coordenadas fuera del territorio peruano:", len(fuera_peru))
print("IIEE sin coordenadas:", padron["NLAT_IE"].isna().sum())
print("Fechas de registro en el futuro:", (padron["FECHAREG"] > HOY).sum())
print("Registros actualizados antes de haber sido creados:",
      (padron["FECHA_ACT"] < padron["FECHAREG"]).sum())

Coordenadas fuera del territorio peruano: 0
IIEE sin coordenadas: 1302
Fechas de registro en el futuro: 1
Registros actualizados antes de haber sido creados: 14


In [36]:
# Coordenadas fuera del país -> NaN; fechas imposibles -> NaT
padron.loc[fuera_peru.index, ["NLAT_IE", "NLONG_IE"]] = np.nan
padron.loc[padron["FECHAREG"] > HOY, "FECHAREG"] = pd.NaT

# Imputación de coordenadas con la mediana del distrito (CODGEO)
mediana_distrito = padron.groupby("CODGEO")[["NLAT_IE", "NLONG_IE"]].transform("median")
padron["coord_imputada"] = padron["NLAT_IE"].isna() & mediana_distrito["NLAT_IE"].notna()
padron["NLAT_IE"] = padron["NLAT_IE"].fillna(mediana_distrito["NLAT_IE"])
padron["NLONG_IE"] = padron["NLONG_IE"].fillna(mediana_distrito["NLONG_IE"])

print("Coordenadas imputadas con la mediana distrital:", int(padron["coord_imputada"].sum()))
print("IIEE que siguen sin coordenadas:", int(padron["NLAT_IE"].isna().sum()))

Coordenadas imputadas con la mediana distrital: 1302
IIEE que siguen sin coordenadas: 0


### <font color=skyblue> 4.5 Filtrado a primaria y eliminación de duplicados — Padron_web </font>

El padrón contiene **todos** los niveles educativos del país (inicial, primaria, secundaria,
técnico productiva...). Como el trabajo es sobre primaria, se filtra antes de integrar: así
el cruce no arrastra 130 mil filas que nunca van a encontrar pareja.

In [37]:
print("Niveles en el padrón:")
display(padron["D_NIV_MOD"].value_counts())

padron_primaria = padron[padron["D_NIV_MOD"] == "Primaria"].copy()
print("\nPadrón filtrado a primaria:", padron_primaria.shape)

print("IDs repetidos antes:", padron_primaria.duplicated(subset=LLAVE).sum())
padron_limpio = padron_primaria.drop_duplicates(subset=LLAVE, keep="first")
print("Padrón primaria sin duplicados:", padron_limpio.shape)

Niveles en el padrón:


,count
D_NIV_MOD,
Inicial - Programa no escolarizado,55822
Primaria,48061
Inicial - Jardín,41455
Secundaria,19323
Técnico Productiva,2715
Inicial - Cuna-jardín,2654
Educación Ocupacional,2365
Básica Alternativa-Avanzado,1820
Instituto Superior Tecnológico,1542



Padrón filtrado a primaria: (48061, 46)
IDs repetidos antes: 0
Padrón primaria sin duplicados: (48061, 46)


In [38]:
# Nos quedamos con las columnas útiles para el análisis
columnas_padron = ["COD_MOD", "ANEXO", "CODLOCAL", "CEN_EDU", "D_NIV_MOD", "D_GESTION",
                   "D_GES_DEP", "D_TIPSSEXO", "DAREACENSO", "CODGEO", "D_DPTO", "D_PROV",
                   "D_DIST", "D_REGION", "D_DREUGEL", "NLAT_IE", "NLONG_IE", "D_ESTADO",
                   "coord_imputada"]

padron_limpio = padron_limpio[columnas_padron]
display(padron_limpio.head())

,COD_MOD,ANEXO,CODLOCAL,CEN_EDU,D_NIV_MOD,D_GESTION,D_GES_DEP,D_TIPSSEXO,DAREACENSO,CODGEO,D_DPTO,D_PROV,D_DIST,D_REGION,D_DREUGEL,NLAT_IE,NLONG_IE,D_ESTADO,coord_imputada
22,0411504,0,016751,COLEGIO PARROQUIAL NUESTRA SEÑORA DEL SAGRADO ...,Primaria,Privada,Comunidad o asociación religiosa,Mixto,Urbana,020105,ANCASH,HUARAZ,INDEPENDENCIA,DRE ANCASH,UGEL HUARAZ,-9.515578,-77.532393,Activo,False
23,0411512,0,015676,FE Y ALEGRIA 19,Primaria,Pública de gestión privada,Convenio con Sector Educación,Mixto,Urbana,020101,ANCASH,HUARAZ,HUARAZ,DRE ANCASH,UGEL HUARAZ,-9.535106,-77.520957,Activo,False
24,0411678,0,016218,GRAN MARISCAL TORIBIO DE LUZURIAGA,Primaria,Pública de gestión directa,Sector Educación,Mixto,Urbana,020105,ANCASH,HUARAZ,INDEPENDENCIA,DRE ANCASH,UGEL HUARAZ,-9.514930,-77.530520,Activo,False
25,0411728,0,015370,86019 LA LIBERTAD,Primaria,Pública de gestión directa,Sector Educación,Mixto,Urbana,020101,ANCASH,HUARAZ,HUARAZ,DRE ANCASH,UGEL HUARAZ,-9.528400,-77.525940,Activo,False
26,0415901,0,016242,86021 SIMON ANTONIO BOLIVAR PALACIOS,Primaria,Pública de gestión directa,Sector Educación,Mixto,Urbana,020105,ANCASH,HUARAZ,INDEPENDENCIA,DRE ANCASH,UGEL HUARAZ,-9.519680,-77.533200,Activo,False


## <font color=yellow> 5. Integración de datos </font>

### <font color=skyblue> 5.0 Granularidad: qué es cada fila antes de integrar </font>

Antes de cruzar hay que comprobar que las cuatro tablas estén a la misma granularidad. Dos de ellas no lo estaban en
origen y se agregaron **antes** de publicarlas en Hugging Face (el código está en las secciones 2.1 y 3.4):

| Fuente | Una fila en el archivo original es… | Qué se hizo para llegar a “una fila = un colegio” |
|---|---|---|
| Censo 3AP | un colegio | Nada: ya estaba a nivel colegio |
| Padrón | un servicio educativo de **cualquier nivel** | Filtrar a Primaria y quitar llaves repetidas |
| Matrícula (C201) | un colegio **en un turno** | Sumar los turnos por colegio (`n_turnos` guarda cuántos tenía) |
| Resultado (3BP) | colegio × cuadro × situación final | Pivotear: una columna por situación (promovido, repite, retirado…) |

**Relación esperada entre tablas (diagrama E-R simplificado):**

```
                 1 : 1
  PADRÓN primaria ─────── CENSO 3AP  ──(inner)──►  BASE ESCOLAR
                                                      │
                                     1 : 0..1 (left)  ├──── MATRÍCULA   (turnos sumados)
                                     1 : 0..1 (left)  └──── RESULTADO   (pivoteado)
```

Como la cardinalidad esperada es **1 a 1**, cada `merge` usa `validate="one_to_one"`: si alguna llave estuviera
repetida, pandas detiene el proceso con un error en lugar de duplicar filas en silencio.

In [39]:
# Filas y unicidad de la llave de cada fuente, ya preparada
granularidad = pd.DataFrame([
    ["Censo 3AP", len(df_censo_final), df_censo_final.duplicated(subset=LLAVE).sum()],
    ["Padrón (todos los niveles)", len(padron), padron.duplicated(subset=LLAVE).sum()],
    ["Padrón primaria sin duplicados", len(padron_limpio), padron_limpio.duplicated(subset=LLAVE).sum()],
    ["Matrícula (turnos sumados)", len(df_matricula), df_matricula.duplicated(subset=LLAVE).sum()],
    ["Resultado (pivoteado)", len(df_resultado), df_resultado.duplicated(subset=LLAVE).sum()],
], columns=["fuente", "filas", "llaves_repetidas"])

display(granularidad)

,fuente,filas,llaves_repetidas
0,Censo 3AP,37354,0
1,Padrón (todos los niveles),180792,0
2,Padrón primaria sin duplicados,48061,0
3,Matrícula (turnos sumados),38338,0
4,Resultado (pivoteado),38036,0


### <font color=skyblue> 5.1 Estrategia de integración </font>

Las cuatro fuentes comparten la misma unidad de análisis y la misma llave, así que la
integración es directa. El orden y el tipo de cruce sí importan:

1.  **Censo + Padrón** → `inner`. Solo interesan los colegios de primaria que existen en
    ambas fuentes: el censo aporta las respuestas del director y el padrón la ubicación
    y el tipo de gestión. Un colegio que falte en cualquiera de las dos no es analizable.
2.  **+ Matrícula** → `left`. La matrícula es el denominador de las tasas, así que se
    adjunta a **todos** los colegios de la base escolar de forma independiente.
3.  **+ Resultado del ejercicio** → `left`. Se conservan todos los colegios y se les adjunta
    su resultado si existe; los que no tengan quedan con NaN, que es información honesta.

Un orden alternativo sería cruzar primero resultado y matrícula (`inner`) y después unir ese
bloque a la base escolar. Se descartó porque cualquier colegio con matrícula pero sin resultado
perdería su denominador sin necesidad. Aquí la matrícula y el resultado se incorporan por separado.

En la versión anterior del trabajo estos cruces eran `outer`, lo que creaba filas sin colegio real y
obligaba a eliminarlas después con un umbral arbitrario de NaN. Aquí no hace falta.

### <font color=skyblue> 5.2 Integración de la base escolar (Censo + Padrón) </font>

In [40]:
base_escolar = pd.merge(
    df_censo_final,
    padron_limpio,
    on=LLAVE,
    how="inner",
    suffixes=("_censo", "_padron"),
    indicator="cruce_censo_padron",
    validate="one_to_one",      # detiene el proceso si alguna llave se repite
)

print("Censo:", df_censo_final.shape[0], "colegios")
print("Padrón primaria:", padron_limpio.shape[0], "colegios")
print("Base escolar integrada:", base_escolar.shape)
display(base_escolar[["COD_MOD", "CEN_EDU_censo", "DISTRITO", "D_DPTO",
                      "D_GESTION", "DAREACENSO"]].head(10))

Censo: 37354 colegios
Padrón primaria: 48061 colegios
Base escolar integrada: (37354, 51)


,COD_MOD,CEN_EDU_censo,DISTRITO,D_DPTO,D_GESTION,DAREACENSO
0,1720556,Sir Alexander Fleming,Cercado,AREQUIPA,Privada,Urbana
1,0313890,40513 San Miguel Arcangel,Alca,AREQUIPA,Pública de gestión directa,Rural
2,1495092,33421,Pillco Marca,HUANUCO,Pública de gestión directa,Urbana
3,1579861,Santander De Puente Piedra,Puente Piedra,LIMA,Privada,Urbana
4,1529767,Colegio Superior De Ciencias Newton Vianney,Yura,AREQUIPA,Privada,Urbana
5,0358028,88154,Cabana,ANCASH,Pública de gestión directa,Rural
6,1374362,17566,Sallique,CAJAMARCA,Pública de gestión directa,Rural
7,0925248,30001-54,Satipo,JUNIN,Pública de gestión directa,Urbana
8,0215004,30566 Alejandro Rufino Espinoza Leon,Yauyos,JUNIN,Pública de gestión directa,Rural
9,0287219,43064,La Capilla,MOQUEGUA,Pública de gestión directa,Rural


In [41]:
# ¿Cuántos colegios del censo no encontraron pareja en el padrón?
solo_censo = len(df_censo_final) - len(base_escolar)
print(f"Colegios del censo sin correspondencia en el padrón: {solo_censo}")
print(f"Porcentaje de pérdida: {solo_censo / len(df_censo_final) * 100:.2f}%")

# Y en la otra dirección: colegios de primaria del padrón que no están en el censo 3AP
solo_padron = padron_limpio[~padron_limpio.set_index(LLAVE).index.isin(base_escolar.set_index(LLAVE).index)]
print(f"\nColegios del padrón de primaria sin cédula 3AP: {len(solo_padron)}")
print("Estado de esos colegios en el padrón:")
display(solo_padron["D_ESTADO"].value_counts())

Colegios del censo sin correspondencia en el padrón: 0
Porcentaje de pérdida: 0.00%

Colegios del padrón de primaria sin cédula 3AP: 10707
Estado de esos colegios en el padrón:


,count
D_ESTADO,
Inactivo,9055
Activo,1652


**Decisión sobre los no emparejados.** Todos los colegios del censo encuentran su pareja en el padrón. En la otra
dirección, 10,707 colegios de primaria del padrón no están en el censo, y **9,055 de ellos (85%) figuran como
inactivos**: son servicios cerrados o sin funcionamiento en 2025, lo que explica la mayor parte de la diferencia. Se
excluyen porque sin la cédula 3AP no tienen ninguna de las variables de convivencia, que son el centro del problema.

### <font color=skyblue> 5.3 Incorporación de la matrícula (denominador) </font>

In [42]:
# Se descartan las columnas de contexto que la matrícula repite y que ya vienen
# del censo y del padrón, para no generar sufijos innecesarios.
contexto_matricula = ["NIV_MOD", "GES_DEP", "AREA_CENSO", "CODGEO", "CODOOII", "CODLOCAL"]

base_con_matricula = pd.merge(
    base_escolar,
    df_matricula.drop(columns=contexto_matricula),
    on=LLAVE,
    how="left",
    indicator="cruce_matricula",
    validate="one_to_one",
)

print("Colegios de la base escolar:", base_escolar.shape[0])
print("Base con matrícula:", base_con_matricula.shape)
display(base_con_matricula["cruce_matricula"].value_counts())

# Registros de matrícula que no cruzan con la base escolar
llaves_base = base_escolar.set_index(LLAVE).index
print("Colegios con matrícula que no están en la base escolar:",
      (~df_matricula.set_index(LLAVE).index.isin(llaves_base)).sum())

Colegios de la base escolar: 37354
Base con matrícula: (37354, 76)


,count
cruce_matricula,
both,37354
left_only,0
right_only,0


Colegios con matrícula que no están en la base escolar: 984


### <font color=skyblue> 5.4 Incorporación de los resultados — base final </font>

In [43]:
contexto_resultado = ["NIV_MOD", "GES_DEP", "AREA_CENSO", "CODGEO", "CODOOII", "FUENTE"]

base_final = pd.merge(
    base_con_matricula,
    df_resultado.drop(columns=contexto_resultado),
    on=LLAVE,
    how="left",
    indicator="cruce_con_resultados",
    validate="one_to_one",
)

print("Base final:", base_final.shape)
display(base_final["cruce_con_resultados"].value_counts())

# La llave debe seguir siendo única: si esto no da 0, algún merge duplicó filas
print("\nIDs repetidos en la base final:", base_final.duplicated(subset=LLAVE).sum())

# ¿Qué colegios no tienen resultado del ejercicio?
sin_resultado = base_final[base_final["cruce_con_resultados"] == "left_only"]
print("\nColegios sin resultado, por gestión:")
display(sin_resultado["D_GESTION"].value_counts())

print("Colegios con resultado que no están en la base escolar:",
      (~df_resultado.set_index(LLAVE).index.isin(llaves_base)).sum())

Base final: (37354, 110)


,count
cruce_con_resultados,
both,37232
left_only,122
right_only,0



IDs repetidos en la base final: 0

Colegios sin resultado, por gestión:


,count
D_GESTION,
Privada,104
Pública de gestión directa,17
Pública de gestión privada,1


Colegios con resultado que no están en la base escolar: 804


**Lectura de la validación.**

*   Ningún `merge` duplicó filas: la base final tiene exactamente un registro por colegio (IDs repetidos = 0).
*   Todos los colegios de la base tienen matrícula, así que ninguno pierde su denominador.
*   122 colegios de la base (0.3%, casi todos privados) no tienen resultado del ejercicio. **Se conservan** con NaN en
    repetición y retiro: siguen siendo útiles para el análisis del registro de violencia y solo quedan fuera de los
    cálculos de trayectoria escolar.
*   984 colegios con matrícula y 804 con resultado no están en la base escolar. **No se incorporan**: el `left` los
    excluye a propósito, porque no tienen respuestas del censo 3AP.

### <font color=skyblue> 5.5 Construcción de indicadores </font>

Los conteos absolutos no son comparables entre colegios de distinto tamaño. Se
construyen tasas usando la matrícula como denominador.

Antes de dividir hacen falta dos resguardos:

*   **Coherencia.** Un colegio no puede registrar más incidencias que estudiantes tiene
    matriculados. Esos casos son un error de captura y se anulan.
*   **Tamaño mínimo.** En un colegio con 5 estudiantes, una sola incidencia produce una
    tasa de 20 por cada 100, que no es comparable con la de un colegio de 500. Las tasas
    se calculan solo para colegios con al menos 10 estudiantes (`MIN_MATRICULA`); en los
    gráficos de repetición se exigen 20 evaluados (`MIN_EVALUADOS`).

**Índice de gestión de la convivencia.** Suma un punto por cada elemento de gestión presente:
responsable de convivencia (`P123B`), conocimiento de los protocolos de atención (`P128B`) y
conocimiento de la obligación de reportar a la UGEL (`P129B`). **No incluye el libro de
incidencias** (`P125B`) a propósito: el libro es el instrumento con el que se mide el registro,
así que incluirlo haría circular el análisis (un colegio sin libro casi no puede registrar
incidencias por definición). Si el director no respondió alguno de los tres elementos, el índice
queda en NaN en lugar de contarlo como "no tiene".

In [44]:
# Resguardo 1: casos imposibles respecto a la matrícula
for col in ["incidencias_libro", "casos_siseve"]:
    incoherentes = base_final[col] > base_final["mat_total"]
    print(f"{col}: {int(incoherentes.sum())} colegios con más casos que estudiantes matriculados")
    base_final.loc[incoherentes, col] = np.nan

# Resguardo 2: tamaños mínimos para que las tasas sean interpretables
MIN_MATRICULA = 10
MIN_EVALUADOS = 20
print("Colegios con menos de", MIN_MATRICULA, "estudiantes:",
      int((base_final["mat_total"] < MIN_MATRICULA).sum()))

incidencias_libro: 24 colegios con más casos que estudiantes matriculados
casos_siseve: 14 colegios con más casos que estudiantes matriculados
Colegios con menos de 10 estudiantes: 7622


In [45]:
matricula = base_final["mat_total"].replace(0, np.nan)
matricula = matricula.where(matricula >= MIN_MATRICULA)

# Indicadores de violencia escolar (por cada 100 estudiantes)
base_final["tasa_incidencias_100"] = (base_final["incidencias_libro"] / matricula * 100).round(3)
base_final["tasa_casos_siseve_100"] = (base_final["casos_siseve"] / matricula * 100).round(3)

# Indicadores de trayectoria escolar
evaluados = base_final["res_evaluados"].replace(0, np.nan)
base_final["tasa_repeticion"] = (base_final["res_permanece_grado"] / evaluados).round(4)
base_final["tasa_retiro"] = (base_final["res_retirado"] / evaluados).round(4)

# Indicador resumen de gestión de la convivencia escolar (0 a 3)
gestion_convivencia = ["P123B", "P128B", "P129B"]
respondio_todo = base_final[gestion_convivencia].notna().all(axis=1)
base_final["indice_convivencia"] = (
    sum((base_final[c] == "SI").astype(int) for c in gestion_convivencia)
    .where(respondio_todo)
    .astype("Int64")
)

# Tamaño del colegio, como variable de control
base_final["tam_colegio"] = pd.cut(
    base_final["mat_total"],
    bins=[0, 30, 100, 300, 100000],
    labels=["Muy pequeño (1-30)", "Pequeño (31-100)", "Mediano (101-300)", "Grande (300+)"],
)

display(base_final[["mat_total", "incidencias_libro", "tasa_incidencias_100",
                    "res_evaluados", "tasa_repeticion", "tasa_retiro",
                    "indice_convivencia"]].describe().round(4))

,mat_total,incidencias_libro,tasa_incidencias_100,res_evaluados,tasa_repeticion,tasa_retiro,indice_convivencia
count,37354.0000,31453.0000,25587.0000,37232.0000,37232.0000,37232.0000,37303.0
mean,96.9277,0.5163,0.6612,94.7906,0.0148,0.0036,2.6308
std,171.1974,3.0909,3.2972,166.2391,0.0465,0.0244,0.7289
min,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0
25%,12.0000,0.0000,0.0000,12.0000,0.0000,0.0000,3.0
50%,32.0000,0.0000,0.0000,31.0000,0.0000,0.0000,3.0
75%,96.0000,0.0000,0.0000,96.0000,0.0000,0.0000,3.0
max,2234.0000,230.0000,100.0000,2234.0000,1.0000,1.0000,3.0


### <font color=skyblue> 5.6 Verificación final de la base integrada </font>

In [46]:
verificacion = pd.DataFrame({
    "n_faltantes": base_final.isna().sum(),
    "porcentaje": (base_final.isna().mean() * 100).round(2),
})
display(verificacion.sort_values("porcentaje", ascending=False).head(20))

print("Dimensiones finales:", base_final.shape)
print("Colegios con al menos un indicador de violencia:",
      base_final[["incidencias_libro", "casos_siseve"]].notna().any(axis=1).sum())
print("Colegios con datos de repetición:", base_final["tasa_repeticion"].notna().sum())

,n_faltantes,porcentaje
P126B_SI_1,32721,87.60
tasa_incidencias_100,11767,31.50
P106B,10680,28.59
P116A,7714,20.65
tasa_casos_siseve_100,7653,20.49
P124B_SI,6591,17.64
incidencias_libro,5901,15.80
P126B,5875,15.73
P105B,5828,15.60
P122B,2150,5.76


Dimensiones finales: (37354, 114)
Colegios con al menos un indicador de violencia: 37339
Colegios con datos de repetición: 37232


**¿Los faltantes en las incidencias son aleatorios?** Antes de graficar hay que saber *quiénes* no
respondieron la pregunta sobre el libro de incidencias, porque si faltan más en unos grupos que en otros,
las comparaciones entre grupos se calculan con muestras distintas.

In [47]:
base_final["sin_dato_incidencias"] = base_final["incidencias_libro"].isna()

sin_dato = pd.concat([
    base_final.groupby(var, observed=True)["sin_dato_incidencias"].mean().mul(100)
    .rename_axis("categoria").reset_index(name="pct_sin_dato").assign(variable=nombre)
    for var, nombre in [("tam_colegio", "Tamaño"), ("DAREACENSO", "Área"), ("D_GESTION", "Gestión")]
])
display(sin_dato[["variable", "categoria", "pct_sin_dato"]].round(1))

print("\nDepartamentos con más colegios sin dato (%):")
display(base_final.groupby("D_DPTO")["sin_dato_incidencias"].mean().mul(100)
        .sort_values(ascending=False).head(6).round(1))

,variable,categoria,pct_sin_dato
0,Tamaño,Muy pequeño (1-30),20.9
1,Tamaño,Pequeño (31-100),14.4
2,Tamaño,Mediano (101-300),9.0
3,Tamaño,Grande (300+),4.0
0,Área,Rural,18.6
1,Área,Urbana,10.2
0,Gestión,Privada,13.9
1,Gestión,Pública de gestión directa,16.4
2,Gestión,Pública de gestión privada,12.4



Departamentos con más colegios sin dato (%):


,sin_dato_incidencias
D_DPTO,
LORETO,47.0
UCAYALI,36.3
MADRE DE DIOS,22.9
AMAZONAS,21.6
PUNO,19.2
CAJAMARCA,17.7


**Lectura.** La falta de respuesta **no es aleatoria**: es mayor en colegios pequeños y rurales, y muy
alta en Loreto y Ucayali. Es un faltante de tipo *MAR* (depende de características observables), no
*MCAR*. No se imputa, porque inventar incidencias sería peor que reconocer el vacío, pero hay que
tenerlo presente: las tasas de esos departamentos se calculan solo con los colegios que respondieron.

In [48]:
# Se guarda la base integrada (una fila por colegio) para que cualquiera pueda
# partir de aquí sin volver a descargar y limpiar las cuatro fuentes.
columnas_auxiliares = ["cruce_censo_padron", "cruce_matricula", "cruce_con_resultados"]
base_final.drop(columns=columnas_auxiliares).to_parquet(SALIDAS / "base_integrada.parquet", index=False)
print("Base integrada guardada:", base_final.shape, "->", SALIDAS / "base_integrada.parquet")

Base integrada guardada: (37354, 115) -> salidas/base_integrada.parquet


## <font color=yellow> 6. EDA sobre la base integrada </font>

### <font color=skyblue> 6.1 Gráfico 1 — Registro de violencia escolar por departamento </font>

El indicador se calcula como **tasa agregada**: total de casos del departamento entre
total de estudiantes matriculados del departamento. No se usa el promedio de las tasas
individuales porque los colegios muy pequeños generan valores extremos que dominan la media.

In [49]:
def tasa_agregada(grupo, columna):
    """Casos por cada 100 estudiantes, sumando primero y dividiendo después."""
    valido = grupo[columna].notna() & grupo["mat_total"].notna()
    if valido.sum() == 0:
        return np.nan
    return grupo.loc[valido, columna].sum() / grupo.loc[valido, "mat_total"].sum() * 100

por_dpto = (base_final
            .groupby("D_DPTO")
            .apply(lambda g: pd.Series({
                "colegios": len(g),
                "Incidencias en el libro (2024)": tasa_agregada(g, "incidencias_libro"),
                "Casos reportados al SíseVe": tasa_agregada(g, "casos_siseve"),
                "% colegios sin dato de incidencias": g["incidencias_libro"].isna().mean() * 100,
            }))
            .round(3)
            .reset_index())

display(por_dpto.sort_values("Incidencias en el libro (2024)", ascending=False).head(10))

datos_grafico = por_dpto.melt(
    id_vars=["D_DPTO", "colegios"],
    value_vars=["Incidencias en el libro (2024)", "Casos reportados al SíseVe"],
    var_name="Indicador", value_name="Tasa")

fig1 = px.bar(
    datos_grafico, y="D_DPTO", x="Tasa", color="Indicador", orientation="h",
    barmode="group",
    title="Registro de violencia escolar por departamento<br><sup>Tasa agregada por cada 100 estudiantes matriculados — Primaria, 2024-2025</sup>",
    labels={"D_DPTO": "Departamento", "Tasa": "Casos por cada 100 estudiantes"},
    template="plotly_white",
)
fig1.update_layout(yaxis={"categoryorder": "total ascending"}, height=850)
fig1.show()

/tmp/ipykernel_544/3234012436.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,D_DPTO,colegios,Incidencias en el libro (2024),Casos reportados al SíseVe,% colegios sin dato de incidencias
18,PASCO,707.0,0.810,0.197,14.710
15,LORETO,2376.0,0.679,0.068,46.970
24,UCAYALI,842.0,0.661,0.111,36.342
0,AMAZONAS,1294.0,0.659,0.105,21.638
6,CALLAO,496.0,0.647,0.172,11.492
4,AYACUCHO,1387.0,0.636,0.180,8.868
10,ICA,647.0,0.608,0.123,10.665
1,ANCASH,1821.0,0.586,0.204,9.885
7,CUSCO,1853.0,0.581,0.261,10.847
22,TACNA,239.0,0.577,0.461,2.929


In [50]:
# Comparación justa entre las dos fuentes: solo colegios con dato en AMBOS indicadores
comun = base_final.dropna(subset=["incidencias_libro", "casos_siseve", "mat_total"])
por_dpto_comun = comun.groupby("D_DPTO").agg(libro=("incidencias_libro", "sum"),
                                              siseve=("casos_siseve", "sum"))

print(f"Colegios con ambos indicadores: {len(comun)} de {len(base_final)}")
print("Departamentos donde el libro supera al SíseVe:",
      f"{(por_dpto_comun['libro'] > por_dpto_comun['siseve']).sum()} de {len(por_dpto_comun)}")
print(f"Total: {comun['incidencias_libro'].sum():.0f} incidencias en libros vs "
      f"{comun['casos_siseve'].sum():.0f} casos en SíseVe "
      f"(razón {comun['incidencias_libro'].sum() / comun['casos_siseve'].sum():.1f} a 1)")

# Lima: puesto por conteo absoluto vs puesto por tasa
absoluto = base_final.groupby("D_DPTO")["incidencias_libro"].sum().rank(ascending=False)
por_tasa = por_dpto.set_index("D_DPTO")["Incidencias en el libro (2024)"].rank(ascending=False)
print(f"Lima: puesto {absoluto['LIMA']:.0f} por conteo absoluto, puesto {por_tasa['LIMA']:.0f} por tasa (de {len(por_tasa)})")

Colegios con ambos indicadores: 31420 de 37354
Departamentos donde el libro supera al SíseVe: 25 de 25
Total: 16223 incidencias en libros vs 6187 casos en SíseVe (razón 2.6 a 1)
Lima: puesto 1 por conteo absoluto, puesto 16 por tasa (de 25)


**Conclusión**

1.  Al normalizar por matrícula, el ordenamiento cambia por completo respecto a lo que
    mostraría un conteo absoluto. Lima ocupa el puesto 1 del país por número de incidencias,
    pero por tasa queda en el puesto 16 de 25: ese volumen se explica sobre todo por su
    tamaño. Las tasas más altas corresponden a Pasco y a tres departamentos amazónicos
    (Loreto, Ucayali y Amazonas).
2.  Entre las incidencias anotadas en el libro y los casos reportados al SíseVe hay una
    brecha consistente: **en los 25 departamentos** el libro supera al portal, también al
    comparar solo los colegios que tienen dato en ambos indicadores (razón de 2.6 a 1).
    Parte de la brecha es esperable, porque los dos instrumentos no miden exactamente lo
    mismo, pero su tamaño y su consistencia respaldan la decisión de no usar el SíseVe como
    única fuente.
3.  **Precaución con Loreto y Ucayali.** El 47% y el 36% de sus colegios no respondió la
    pregunta sobre el libro de incidencias (ver 5.6), así que sus tasas se calculan con una
    parte de los colegios. Su ubicación en el ranking es menos firme que la de departamentos
    con respuesta casi completa.

### <font color=skyblue> 6.2 Gráfico 2 — Gestión de la convivencia y registro de violencia </font>

Aquí se mide **qué proporción de colegios llega a registrar al menos una incidencia**,
en lugar del promedio de incidencias. La razón es que la distribución de incidencias está
muy concentrada en cero: más del 80% de los colegios reporta ninguna, de modo que el
promedio es inestable y se deja arrastrar por unos pocos valores altos. La proporción de
colegios que registra es un indicador mucho más estable de capacidad institucional.

In [51]:
por_indice = (base_final
              .dropna(subset=["incidencias_libro", "indice_convivencia"])
              .groupby("indice_convivencia")
              .agg(colegios=("COD_MOD", "size"),
                   pct_registra_incidencias=("incidencias_libro", lambda s: (s > 0).mean() * 100),
                   pct_afiliado_siseve=("P124B", lambda s: (s == "SI").mean() * 100))
              .round(2)
              .reset_index())

display(por_indice)

datos_indice = por_indice.melt(
    id_vars=["indice_convivencia", "colegios"],
    value_vars=["pct_registra_incidencias", "pct_afiliado_siseve"],
    var_name="Indicador", value_name="Porcentaje")

datos_indice["Indicador"] = datos_indice["Indicador"].replace({
    "pct_registra_incidencias": "Registra al menos una incidencia",
    "pct_afiliado_siseve": "Está afiliado al SíseVe",
})

fig2 = px.bar(
    datos_indice, x="indice_convivencia", y="Porcentaje", color="Indicador",
    barmode="group", text="Porcentaje",
    title="Capacidad de registro según el índice de gestión de la convivencia<br><sup>El índice suma 1 punto por cada elemento presente: responsable de convivencia, conocimiento de protocolos y de la obligación de reportar a la UGEL</sup>",
    labels={"indice_convivencia": "Índice de gestión de la convivencia (0 a 3)",
            "Porcentaje": "% de colegios"},
    template="plotly_white",
)
fig2.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig2.update_layout(height=520)
fig2.update_xaxes(type="category")
fig2.show()

,indice_convivencia,colegios,pct_registra_incidencias,pct_afiliado_siseve
0,0,233,3.00,36.05
1,1,1172,5.46,61.35
2,2,4275,8.65,75.04
3,3,25738,15.23,92.61


**Conclusión**

1.  Ambos indicadores crecen de forma monótona con el índice. Entre los colegios con
    índice 0, solo el 3% registra al menos una incidencia y el 36% está afiliado al SíseVe;
    entre los de índice 3, el 15% registra y el 93% está afiliado. Como el libro de
    incidencias no forma parte del índice, este gradiente no es circular.
2.  Esto no significa que haya más violencia donde hay mejor gestión. La lectura correcta
    es de **subregistro**: un colegio sin responsable de convivencia y cuyo director no
    conoce los protocolos no registra casos porque no tiene con qué registrarlos. Su cero
    es un cero de capacidad institucional, no de ausencia de violencia.
3.  Queda una duda: los colegios con mejor gestión también son más grandes (correlación de
    Spearman de 0.19 entre el índice y la matrícula). La sección 6.3 separa ambos efectos.

### <font color=skyblue> 6.3 Gráfico 3 — ¿Qué factores se asocian al registro? (controlando por tamaño) </font>

El gráfico anterior deja una duda: los colegios con mejor gestión también tienden a ser más grandes, y un colegio
grande tiene más estudiantes y por lo tanto más ocasiones de registrar una incidencia. Para saber qué pesa
realmente se hacen dos cosas:

1.  **Descriptivo.** Se separa cada factor por tamaño de colegio: si el índice de gestión importa de verdad,
    la línea debe subir *dentro de cada tamaño*, no solo en el total.
2.  **Regresión logística.** Estima el efecto de cada factor **manteniendo constantes los demás**. El resultado
    se lee como *odds ratio* (OR): un OR de 1.5 significa que las chances (odds) de registrar se multiplican por 1.5
    cuando ese factor está presente; un OR de 1 significa que no hay efecto.

In [52]:
# Base de análisis: colegios que respondieron la pregunta del libro de incidencias
base_reg = base_final.dropna(subset=["incidencias_libro", "mat_total"]).copy()
base_reg["registra"] = (base_reg["incidencias_libro"] > 0).astype(int)

# % de colegios que registra al menos una incidencia, según cada característica
factores = pd.concat([
    base_reg.groupby(var, observed=True)["registra"]
    .agg(colegios="size", pct_registra=lambda s: s.mean() * 100)
    .rename_axis("categoria").reset_index().assign(factor=nombre)
    for var, nombre in [("tam_colegio", "Tamaño"), ("DAREACENSO", "Área"), ("D_GESTION", "Gestión")]
])
display(factores[["factor", "categoria", "colegios", "pct_registra"]].round(1))

# ¿El índice de gestión está asociado al tamaño?
print("Correlación de Spearman entre índice de gestión y matrícula:",
      round(base_final["indice_convivencia"].astype(float).corr(base_final["mat_total"], method="spearman"), 2))

,factor,categoria,colegios,pct_registra
0,Tamaño,Muy pequeño (1-30),14374,5.3
1,Tamaño,Pequeño (31-100),8718,11.5
2,Tamaño,Mediano (101-300),5183,22.9
3,Tamaño,Grande (300+),3178,44.8
0,Área,Rural,20382,8.5
1,Área,Urbana,11071,23.9
0,Gestión,Privada,6744,15.5
1,Gestión,Pública de gestión directa,24326,13.3
2,Gestión,Pública de gestión privada,383,23.8


Correlación de Spearman entre índice de gestión y matrícula: 0.19


In [53]:
# El índice de gestión, separado por tamaño del colegio
por_indice_tam = (base_reg.dropna(subset=["indice_convivencia"])
                  .groupby(["tam_colegio", "indice_convivencia"], observed=True)["registra"]
                  .agg(colegios="size", pct_registra=lambda s: s.mean() * 100)
                  .reset_index())
# Se omiten celdas con menos de 30 colegios: un porcentaje sobre 5 colegios no es confiable
por_indice_tam = por_indice_tam[por_indice_tam["colegios"] >= 30]

display(por_indice_tam.pivot(index="indice_convivencia", columns="tam_colegio", values="pct_registra").round(1))

fig3 = px.line(
    por_indice_tam, x="indice_convivencia", y="pct_registra", color="tam_colegio", markers=True,
    title="Registro de incidencias según el índice de gestión, por tamaño del colegio<br><sup>Si el índice solo reflejara tamaño, cada línea sería plana</sup>",
    labels={"indice_convivencia": "Índice de gestión de la convivencia (0 a 3)",
            "pct_registra": "% de colegios que registra al menos una incidencia",
            "tam_colegio": "Tamaño del colegio"},
    template="plotly_white",
)
fig3.update_xaxes(type="category")
fig3.update_layout(height=500)
fig3.show()

tam_colegio,Muy pequeño (1-30),Pequeño (31-100),Mediano (101-300),Grande (300+)
indice_convivencia,,,,
0,2.2,5.3,NaN,NaN
1,3.3,6.3,12.9,NaN
2,4.5,10.4,17.6,31.5
3,5.6,11.9,23.8,45.8


In [54]:
# Regresión logística: ¿qué factores se asocian a registrar, con los demás constantes?
base_reg["log2_matricula"] = np.log2(base_reg["mat_total"])       # por cada vez que se DUPLICA la matrícula
base_reg["urbano"] = (base_reg["DAREACENSO"] == "Urbana").astype(int)
base_reg["privada"] = (base_reg["D_GESTION"] == "Privada").astype(int)

# SI -> 1, NO -> 0; el NaN se conserva y el modelo descarta esas filas
binarias = {"P123B": "resp_convivencia", "P128B": "conoce_protocolos", "P129B": "reporta_ugel",
            "P124B": "afiliado_siseve", "P116A": "actividad_esi", "P127B": "consulta_rnssc"}
for original, nueva in binarias.items():
    base_reg[nueva] = base_reg[original].map({"SI": 1, "NO": 0})

formula = ("registra ~ log2_matricula + urbano + privada + resp_convivencia + conoce_protocolos"
           " + reporta_ugel + afiliado_siseve + actividad_esi + consulta_rnssc")
modelo = smf.logit(formula, data=base_reg).fit(disp=0)

odds = pd.DataFrame({
    "odds_ratio": np.exp(modelo.params),
    "ic95_inf": np.exp(modelo.conf_int()[0]),
    "ic95_sup": np.exp(modelo.conf_int()[1]),
    "p_valor": modelo.pvalues,
}).drop("Intercept").round(3)

print(f"Colegios usados en el modelo: {int(modelo.nobs)} de {len(base_reg)}")
display(odds.sort_values("odds_ratio", ascending=False))

Colegios usados en el modelo: 26360 de 31453


,odds_ratio,ic95_inf,ic95_sup,p_valor
log2_matricula,1.596,1.552,1.641,0.000
resp_convivencia,1.585,1.274,1.972,0.000
conoce_protocolos,1.356,1.098,1.676,0.005
actividad_esi,1.266,1.173,1.366,0.000
urbano,1.136,1.006,1.283,0.040
afiliado_siseve,0.947,0.818,1.096,0.463
consulta_rnssc,0.946,0.877,1.021,0.156
reporta_ugel,0.865,0.734,1.018,0.081
privada,0.760,0.683,0.845,0.000


**Conclusión**

1.  **El tamaño es el factor más fuerte.** El porcentaje de colegios que registra al menos
    una incidencia pasa de 5.3% en los muy pequeños a 44.8% en los grandes, y de 8.5% en el
    área rural a 23.9% en la urbana.
2.  **El índice de gestión sigue importando dentro de cada tamaño.** En el gráfico, la línea de
    cada tamaño sube con el índice: por ejemplo, de 2.2% a 5.6% entre los colegios muy pequeños
    y de 31.5% a 45.8% entre los grandes. No es solo un efecto de tamaño.
3.  **Regresión logística** (con los demás factores constantes). Cada vez que se duplica la
    matrícula, las chances (odds) de registrar se multiplican por 1.6. Contar con un responsable de
    convivencia las multiplica por 1.6, conocer los protocolos de atención por 1.4 y haber hecho
    actividades de ESI con familias por 1.3; los tres son significativos. La gestión privada se asocia
    con **menos** registro que la pública (OR 0.76) a igual tamaño y gestión de la convivencia. La
    afiliación al SíseVe, la consulta al RNSSC y el conocimiento de la obligación de reportar a la
    UGEL no aportan una diferencia distinguible de cero una vez controlado lo anterior.
4.  Respuesta al problema planteado: los factores institucionales asociados al registro son,
    en este orden, el **tamaño del colegio**, la **presencia de un responsable de convivencia**, el
    **conocimiento de protocolos** y el **trabajo en ESI con familias**. Son factores de capacidad
    para registrar, no de cantidad de violencia.
5.  Precauciones: son asociaciones, no causas; el modelo usa 26,360 de los 31,453 colegios con
    dato de incidencias (se pierden los que no respondieron alguna de las preguntas explicativas);
    y no controla por departamento.

### <font color=skyblue> 6.4 Gráfico 4 — Distribución de la repetición según gestión y área </font>

In [55]:
datos_caja = base_final.dropna(subset=["tasa_repeticion", "D_GESTION"]).copy()

# Se excluyen los colegios muy pequeños: con 5 estudiantes evaluados, un solo
# repitente produce una tasa de 20% que no es comparable con la de un colegio grande.
datos_caja = datos_caja[datos_caja["res_evaluados"] >= MIN_EVALUADOS]

fig4 = px.box(
    datos_caja, x="D_GESTION", y="tasa_repeticion", color="DAREACENSO",
    title="Distribución de la tasa de repetición según gestión y área<br><sup>Solo colegios con 20 o más estudiantes evaluados — Primaria, 2025</sup>",
    labels={"D_GESTION": "Tipo de gestión", "tasa_repeticion": "Tasa de repetición",
            "DAREACENSO": "Área"},
    template="plotly_white",
)
fig4.update_layout(height=550, yaxis_tickformat=".0%")
fig4.show()

# La tasa agregada (total de repitentes / total de evaluados) es el indicador principal;
# la media de tasas se muestra solo para comparar. Igual que en 6.1, los colegios pequeños
# distorsionan el promedio simple.
resumen_caja = (datos_caja
                .groupby(["D_GESTION", "DAREACENSO"])
                .agg(colegios=("COD_MOD", "size"),
                     repitentes=("res_permanece_grado", "sum"),
                     evaluados=("res_evaluados", "sum"),
                     media_de_tasas=("tasa_repeticion", "mean"),
                     mediana=("tasa_repeticion", "median"),
                     pct_con_repitente=("res_permanece_grado", lambda s: (s > 0).mean() * 100)))
resumen_caja["tasa_agregada_pct"] = resumen_caja["repitentes"] / resumen_caja["evaluados"] * 100

print("Validación estadística:")
display(resumen_caja[["colegios", "tasa_agregada_pct", "media_de_tasas", "mediana",
                      "pct_con_repitente"]].round(3))

# Concentración: ¿qué porcentaje de colegios acumula la mitad de los repitentes?
repitentes = datos_caja["res_permanece_grado"].sort_values(ascending=False)
pct_colegios_mitad = (repitentes.cumsum() <= repitentes.sum() * 0.5).mean() * 100
print(f"\nEl {pct_colegios_mitad:.1f}% de los colegios acumula el 50% de los repitentes")

# Valores extremos: se revisan, pero no se eliminan (detectar no es lo mismo que eliminar)
extremos = datos_caja[datos_caja["tasa_repeticion"] > 0.4]
print(f"Colegios con tasa de repetición mayor a 40%: {len(extremos)} de {len(datos_caja)} "
      f"(suman {extremos['res_permanece_grado'].sum():.0f} de {repitentes.sum():.0f} repitentes)")
display(extremos[["CEN_EDU_censo", "D_DPTO", "D_GESTION", "DAREACENSO", "res_evaluados",
                  "res_permanece_grado", "tasa_repeticion"]].sort_values("tasa_repeticion", ascending=False))

Validación estadística:


colegios  tasa_agregada_pct  \
D_GESTION                  DAREACENSO                                
Privada                    Rural             91              0.641   
                           Urbana          6873              0.153   
Pública de gestión directa Rural          11670              2.113   
                           Urbana          4271              1.499   
Pública de gestión privada Rural            110              1.670   
                           Urbana           284              0.687   

                                       media_de_tasas  mediana  \
D_GESTION                  DAREACENSO                            
Privada                    Rural                0.008    0.000   
                           Urbana               0.003    0.000   
Pública de gestión directa Rural                0.022    0.000   
                           Urbana               0.017    0.008   
Pública de gestión privada Rural                0.021    0.000   
                           Urbana               0.007    0.000   

                                       pct_con_repitente  
D_GESTION                  DAREACENSO                     
Privada                    Rural                  19.780  
                           Urbana                 12.411  
Pública de gestión directa Rural                  33.179  
                           Urbana                 71.857  
Pública de gestión privada Rural                  29.091  
                           Urbana                 37.676


El 4.6% de los colegios acumula el 50% de los repitentes
Colegios con tasa de repetición mayor a 40%: 11 de 23299 (suman 176 de 41810 repitentes)


,CEN_EDU_censo,D_DPTO,D_GESTION,DAREACENSO,res_evaluados,res_permanece_grado,tasa_repeticion
7873,64783-B,UCAYALI,Pública de gestión directa,Rural,28.0,25.0,0.8929
17485,62838,LORETO,Pública de gestión directa,Rural,32.0,18.0,0.5625
31670,62846,LORETO,Pública de gestión directa,Rural,38.0,20.0,0.5263
5869,60901,LORETO,Pública de gestión directa,Rural,25.0,12.0,0.4800
24317,62419,LORETO,Pública de gestión directa,Rural,31.0,14.0,0.4516
25179,Ie 20987 Victor Raul Haya De La Torre,LIMA,Pública de gestión directa,Urbana,31.0,14.0,0.4516
1111,64705-B,UCAYALI,Pública de gestión directa,Rural,21.0,9.0,0.4286
32696,34429,PASCO,Pública de gestión directa,Rural,28.0,12.0,0.4286
22614,62319,LORETO,Pública de gestión directa,Rural,26.0,11.0,0.4231
36703,62291,LORETO,Pública de gestión directa,Rural,59.0,24.0,0.4068


**Conclusión**

1.  La repetición es sistemáticamente mayor en la gestión pública que en la privada. Con la tasa
    agregada, la pública urbana llega a 1.5% y la privada urbana a 0.15%, es decir, unas diez veces
    menos. La pública rural es la más alta, con 2.1%.
2.  **La diferencia entre rural y urbana en la pública hay que leerla con cuidado.** Por tasa agregada
    la rural es mayor (2.1% frente a 1.5%), pero su mediana es 0 y el 67% de sus colegios no tiene
    ningún repitente, frente al 28% en la urbana. La repetición rural está concentrada en pocos
    colegios, y por eso el diagrama de cajas muestra un cuerpo aplastado en cero con una cola larga.
3.  La concentración es general: el 4.6% de los colegios acumula la mitad de todos los repitentes.
    El promedio nacional oculta situaciones muy distintas y sugiere focalizar en pocos colegios
    antes que en el sistema completo.
4.  Hay 11 colegios con más de 40% de repetición, 9 de ellos en Loreto y Ucayali. Se
    revisaron y **no se eliminaron**: no hay una regla para corregirlos y suman solo 176 de más de
    41 mil repitentes. Conviene tenerlos presentes al interpretar la cola del gráfico.
5.  A diferencia de la versión anterior de este trabajo, esta comparación sí es válida: cada
    colegio aporta su propio dato de repetición, no un valor replicado de su UGEL.

### <font color=skyblue> 6.5 Gráfico 5 — Relación entre violencia registrada y trayectoria escolar </font>

In [56]:
# ¿Los colegios que registran más violencia tienen también más repetición y retiro?
datos_disp = base_final.dropna(
    subset=["tasa_incidencias_100", "tasa_repeticion", "mat_total"]).copy()
datos_disp = datos_disp[datos_disp["res_evaluados"] >= MIN_EVALUADOS]

# El gráfico muestra solo a los colegios que registraron algo (tasa > 0)
fig5 = px.scatter(
    datos_disp[datos_disp["tasa_incidencias_100"] > 0],
    x="tasa_incidencias_100", y="tasa_repeticion",
    color="D_GESTION", size="mat_total", size_max=28, opacity=0.55,
    hover_data={"CEN_EDU_censo": True, "D_DPTO": True, "mat_total": True},
    title="Violencia registrada y repetición escolar<br><sup>Cada punto es un colegio que registró al menos una incidencia; el tamaño representa su matrícula — Primaria</sup>",
    labels={"tasa_incidencias_100": "Incidencias registradas por cada 100 estudiantes",
            "tasa_repeticion": "Tasa de repetición", "D_GESTION": "Gestión"},
    template="plotly_white",
)
fig5.update_layout(height=600, yaxis_tickformat=".0%")
fig5.show()

# La correlación se calcula con TODOS los colegios (incluidos los que registran 0) y con
# Spearman: las tasas están concentradas en cero y Pearson depende de unos pocos valores altos.
def correlaciones(g):
    return pd.Series({
        "colegios": len(g),
        "rho_con_repeticion": g["tasa_incidencias_100"].corr(g["tasa_repeticion"], method="spearman"),
        "rho_con_retiro": g["tasa_incidencias_100"].corr(g["tasa_retiro"], method="spearman"),
    })

total = correlaciones(datos_disp).to_frame("Todos los colegios").T
por_grupo = datos_disp.groupby(["D_GESTION", "DAREACENSO"]).apply(correlaciones)
por_grupo = por_grupo[por_grupo["colegios"] >= 300]      # grupos con muy pocos colegios no son confiables
por_grupo.index = por_grupo.index.map(" / ".join)

print("Correlación de Spearman entre la tasa de incidencias y la trayectoria escolar:")
display(pd.concat([total, por_grupo]).round(3))

Correlación de Spearman entre la tasa de incidencias y la trayectoria escolar:


/tmp/ipykernel_544/1844011229.py:30: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,colegios,rho_con_repeticion,rho_con_retiro
Todos los colegios,20443.0,0.061,0.095
Privada / Urbana,6027.0,-0.025,0.007
Pública de gestión directa / Rural,9904.0,0.014,0.034
Pública de gestión directa / Urbana,4086.0,-0.017,0.074


**Conclusión**

1.  La relación entre violencia registrada y trayectoria escolar es **muy débil**. Con todos los
    colegios, la correlación de Spearman entre la tasa de incidencias y la de repetición es 0.06,
    y con la de retiro 0.09. Dentro de cada grupo de gestión y área es aún menor para la repetición
    (entre -0.03 y 0.01) y no supera 0.08 para el retiro, por lo que la asociación conjunta se
    explica en buena parte por la composición de los grupos.
2.  Dos decisiones de cálculo importan aquí. La correlación se hace con **todos** los colegios y no
    solo con los que registran algo, porque el gráfico solo muestra a estos últimos y esa restricción
    cambia incluso el signo del resultado. Y se usa **Spearman** en lugar de Pearson, porque las tasas se
    concentran en cero y Pearson depende de unos pocos valores altos.
3.  Que la relación sea débil es un resultado, no un fracaso del análisis. Indica que, con
    los datos disponibles, el registro de violencia escolar no predice por sí solo la
    trayectoria académica del colegio, y que ambas dimensiones responden a factores
    distintos.
4.  Es coherente con lo observado en el gráfico 6.2: mientras el indicador de violencia
    esté mediado por la capacidad de registro del colegio, su poder explicativo sobre
    otros resultados educativos queda limitado.

## <font color=yellow> 7. Transformación de variables </font>

Aquí se prueban las transformaciones vistas en el curso sobre **nuestras** variables. No todas sirvieron: para cada una
se muestra el resultado y se decide si entra a la matriz. Como pide el Hito 2, se aplican solo cuando corresponden.

| Transformación | Pregunta que responde |
|---|---|
| 7.1 Codificación | ¿Cómo convertir SI/NO y categorías en números sin inventar un orden? |
| 7.2 Discretización | ¿Conviene agrupar una variable numérica en tramos? ¿Con qué cortes? |
| 7.3 Transformación logarítmica | ¿Se puede corregir la asimetría de las variables? |
| 7.4 Escalamiento | ¿Qué escalador sirve para comparar colegios por distancia? |
| 7.5 Resumen | Qué se usó, qué se descartó y por qué |

### <font color=skyblue> 7.1 Codificación de variables categóricas </font>

Hay tres tipos de variable categórica en la base, y cada una pide una codificación distinta:

*   **Binaria (SI/NO)** → 1/0. El NaN se conserva, porque “no respondió” no es lo mismo que “NO”.
*   **Nominal** (sin orden, como el tipo de gestión) → **one-hot**: una columna 0/1 por categoría.
*   **Ordinal** (con orden, como el tamaño del colegio) → números que respetan ese orden (1 a 4).

In [57]:
# Binarias SI/NO -> 1/0
binarias_matriz = {"P123B": "resp_convivencia", "P125B": "tiene_libro", "P128B": "conoce_protocolos",
                   "P129B": "reporta_ugel", "P124B": "afiliado_siseve", "P116A": "actividad_esi",
                   "P127B": "consulta_rnssc"}
codificadas = pd.DataFrame({nueva: base_final[original].map({"SI": 1, "NO": 0})
                            for original, nueva in binarias_matriz.items()})

# La media de una variable 0/1 es directamente el % de colegios que responde SI
display(pd.DataFrame({"pct_SI": codificadas.mean() * 100,
                      "sin_respuesta": codificadas.isna().sum()}).round(1))

,pct_SI,sin_respuesta
resp_convivencia,87.1,23
tiene_libro,83.2,40
conoce_protocolos,87.1,25
reporta_ugel,88.8,24
afiliado_siseve,82.3,34
actividad_esi,55.6,7714
consulta_rnssc,44.2,25


In [58]:
# Nominal -> one-hot. Se usan nombres cortos para que las columnas sean legibles.
gestion = base_final["D_GESTION"].map({"Pública de gestión directa": "publica",
                                       "Pública de gestión privada": "publica_convenio",
                                       "Privada": "privada"})
dummies_gestion = pd.get_dummies(gestion, prefix="gestion", dtype=int)
print("One-hot de gestión (colegios por columna):")
display(dummies_gestion.sum())

# Con solo dos categorías, el one-hot crea dos columnas redundantes: una es 1 menos la otra.
dummies_area = pd.get_dummies(base_final["DAREACENSO"].str.lower(), prefix="area", dtype=int)
print("¿area_rural + area_urbana = 1 en todas las filas?",
      (dummies_area.sum(axis=1) == 1).all(), "-> basta con una sola columna: urbano")
urbano = (base_final["DAREACENSO"] == "Urbana").astype(int)

# Ordinal -> 1 a 4 respetando el orden de tamaño
orden_tamano = {"Muy pequeño (1-30)": 1, "Pequeño (31-100)": 2, "Mediano (101-300)": 3, "Grande (300+)": 4}
tam_colegio_ord = base_final["tam_colegio"].astype("object").map(orden_tamano)
display(tam_colegio_ord.value_counts().sort_index().rename("colegios"))

One-hot de gestión (colegios por columna):


,0
gestion_privada,7835
gestion_publica,29082
gestion_publica_convenio,437


¿area_rural + area_urbana = 1 en todas las filas? True -> basta con una sola columna: urbano


,colegios
tam_colegio,
1,18168
2,10181
3,5696
4,3309


**¿Por qué no usar *label encoding* (0, 1, 2) para la gestión?** Porque le daría un orden que no existe: un modelo
leería que la “pública de gestión privada” (1) está a mitad de camino entre la pública (0) y la privada (2), y que la
distancia entre pública y privada es el doble. Con one-hot ninguna categoría es “mayor” que otra.

In [59]:
# Caso 1: one-hot del departamento
dummies_dpto = pd.get_dummies(base_final["D_DPTO"], dtype=int)
participacion = dummies_dpto.mean().sort_values() * 100
print(f"El one-hot del departamento crea {dummies_dpto.shape[1]} columnas.")
print(f"Las 5 más pequeñas agrupan entre {participacion.iloc[0]:.1f}% y {participacion.iloc[4]:.1f}% de los colegios cada una.")
print(f"En promedio, cada columna tiene {(1 - dummies_dpto.values.mean()) * 100:.0f}% de ceros.")

# Caso 2: tipo de alumnado por sexo (D_TIPSSEXO)
print("\nTipo de alumnado (% de colegios):")
display(base_final["D_TIPSSEXO"].value_counts(normalize=True).mul(100).round(1))

El one-hot del departamento crea 25 columnas.
Las 5 más pequeñas agrupan entre 0.5% y 1.3% de los colegios cada una.
En promedio, cada columna tiene 96% de ceros.

Tipo de alumnado (% de colegios):


,proportion
D_TIPSSEXO,
Mixto,99.2
Mujeres,0.5
Varones,0.3


**Decisiones.**

*   **Departamento:** no se expande en la matriz. 25 columnas casi vacías agregan ruido a técnicas por distancia
    (K-means, PCA) sin aportar una idea nueva. Se conserva `D_DPTO` como etiqueta para agrupar resultados y, en el
    próximo paso, para controlar por región dentro del modelo (`C(D_DPTO)` en la fórmula, que hace el one-hot
    internamente solo cuando se necesita).
*   **Tipo de alumnado por sexo:** se excluye. Más del 99% de los colegios son mixtos: una variable casi constante no
    distingue a unos colegios de otros, se codifique como se codifique.

### <font color=skyblue> 7.2 Discretización </font>

Discretizar es agrupar una variable numérica en tramos. Se evaluó para dos variables: la **matrícula** (para controlar
por tamaño) y las **incidencias registradas**. Hay tres formas de elegir los cortes:

*   **Igual ancho** (`pd.cut` con n tramos): tramos del mismo tamaño en la escala de la variable.
*   **Igual frecuencia** (`pd.qcut`, cuantiles): cada tramo tiene la misma cantidad de colegios.
*   **Reglas de negocio**: cortes con significado para el problema.

In [60]:
# Matrícula: comparación de los tres criterios
tramos = pd.DataFrame({
    "igual ancho (pd.cut)": pd.cut(base_final["mat_total"], 4).value_counts(sort=False).values,
    "igual frecuencia (pd.qcut)": pd.qcut(base_final["mat_total"], 4).value_counts(sort=False).values,
    "reglas de negocio (usado)": base_final["tam_colegio"].value_counts(sort=False).values,
}, index=["tramo 1", "tramo 2", "tramo 3", "tramo 4"])

print("Colegios en cada tramo según el criterio:")
display(tramos)

print("Cortes de igual ancho:     ", pd.cut(base_final["mat_total"], 4, retbins=True)[1].round(0))
print("Cortes de igual frecuencia:", pd.qcut(base_final["mat_total"], 4, retbins=True)[1].round(0))
print("Cortes de negocio:          [0, 30, 100, 300, más]")

Colegios en cada tramo según el criterio:


,igual ancho (pd.cut),igual frecuencia (pd.qcut),reglas de negocio (usado)
tramo 1,36075,9653,18168
tramo 2,1165,9154,10181
tramo 3,104,9244,5696
tramo 4,10,9303,3309


Cortes de igual ancho:      [-1.000e+00  5.590e+02  1.118e+03  1.676e+03  2.234e+03]
Cortes de igual frecuencia: [1.000e+00 1.200e+01 3.200e+01 9.600e+01 2.234e+03]
Cortes de negocio:          [0, 30, 100, 300, más]


**Interpretación.**

*   **Igual ancho no sirve:** como la matrícula es muy asimétrica (pocos colegios de más de 1,000 estudiantes), casi
    todos los colegios caen en el primer tramo. La variable resultante no distingue nada.
*   **Igual frecuencia reparte bien, pero sus cortes (12, 32 y 96 estudiantes) salen de la muestra**, no del problema:
    cambiarían con otro año de datos y no se pueden explicar a un especialista de la UGEL.
*   **Se usan las reglas de negocio** (`tam_colegio`: hasta 30, 31-100, 101-300 y más de 300). Los tramos tienen
    sentido educativo (el primero agrupa colegios unidocentes y multigrado pequeños) y siguen teniendo suficientes
    colegios cada uno. Esta variable se usó como control en la sección 6.3.

In [61]:
# Incidencias registradas: los cuantiles fallan
try:
    pd.qcut(base_final["incidencias_libro"], 4)
except ValueError as error:
    print("pd.qcut con 4 tramos falla:", error)

print("\nCuartiles de incidencias_libro:", base_final["incidencias_libro"].quantile([0.25, 0.5, 0.75]).tolist())
print(f"% de colegios con 0 incidencias: {(base_final['incidencias_libro'] == 0).mean() / base_final['incidencias_libro'].notna().mean():.1%}")

# Alternativa: reglas de negocio sobre los colegios que sí registran
mediana_registran = base_final.loc[base_final["incidencias_libro"] > 0, "incidencias_libro"].median()
print(f"Mediana de incidencias entre los colegios que registran algo: {mediana_registran:.0f}")

registra_incidencia = (base_final["incidencias_libro"] > 0).astype(float).where(base_final["incidencias_libro"].notna())
nivel_registro = pd.cut(base_final["incidencias_libro"], bins=[-0.1, 0, 2, np.inf], labels=[0, 1, 2]).astype(float)

display(pd.DataFrame({
    "colegios": nivel_registro.value_counts(dropna=False).sort_index(),
}).rename(index={0.0: "0 = no registra", 1.0: "1 = registra 1-2", 2.0: "2 = registra 3 o más"}))

pd.qcut con 4 tramos falla: Bin edges must be unique: Index([0.0, 0.0, 0.0, 0.0, 230.0], dtype='float64', name='incidencias_libro').
You can drop duplicate edges by setting the 'duplicates' kwarg

Cuartiles de incidencias_libro: [0.0, 0.0, 0.0]
% de colegios con 0 incidencias: 86.1%
Mediana de incidencias entre los colegios que registran algo: 2


,colegios
incidencias_libro,
0 = no registra,27086
1 = registra 1-2,2612
2 = registra 3 o más,1755
NaN,5901


**Interpretación.** Con más del 80% de los colegios en cero, los tres cuartiles valen 0 y `pd.qcut` no puede crear tramos
distintos. Por eso la incidencia se discretiza con reglas:

*   `registra_incidencia` (0/1): si el colegio registró al menos una incidencia. Es la variable que se usó en la
    sección 6.2 y en la regresión logística, porque es estable frente a los pocos colegios con conteos muy altos.
*   `nivel_registro` (0, 1, 2): no registra / registra 1-2 / registra 3 o más. El corte en 2 es la mediana de los
    colegios que sí registran, así que separa un registro ocasional de uno más frecuente.

### <font color=skyblue> 7.3 Transformación logarítmica </font>

La asimetría (*skewness*) mide qué tan cargada hacia un lado está una distribución: 0 es simétrica, y valores mayores a
1 indican una cola larga a la derecha. Muchas técnicas (y el propio z-score) funcionan mejor con variables cercanas a
simétricas. Se prueba el logaritmo sobre las variables numéricas principales. Para las tasas se usa `log1p` (log(1+x)),
porque tienen ceros y el logaritmo de 0 no existe.

In [62]:
def asimetria(serie):
    return round(serie.dropna().skew(), 2)

variables_forma = ["mat_total", "tasa_incidencias_100", "tasa_repeticion", "tasa_retiro"]
forma = pd.DataFrame({
    "% de ceros": [round((base_final[v] == 0).mean() / base_final[v].notna().mean() * 100, 1) for v in variables_forma],
    "asimetría original": [asimetria(base_final[v]) for v in variables_forma],
    "asimetría con log": [asimetria(np.log(base_final["mat_total"]))] +
                         [asimetria(np.log1p(base_final[v])) for v in variables_forma[1:]],
}, index=variables_forma)
display(forma)

,% de ceros,asimetría original,asimetría con log
mat_total,0.0,3.58,0.10
tasa_incidencias_100,83.7,11.03,3.57
tasa_repeticion,75.3,6.58,5.28
tasa_retiro,89.1,19.83,15.14


In [63]:
# Antes y después para la matrícula
matricula_forma = pd.concat([
    pd.DataFrame({"valor": base_final["mat_total"], "escala": "Matrícula original"}),
    pd.DataFrame({"valor": np.log(base_final["mat_total"]), "escala": "log(matrícula)"}),
])
fig_log = px.histogram(matricula_forma, x="valor", facet_col="escala", nbins=60,
                       title="La transformación logarítmica corrige la asimetría de la matrícula",
                       template="plotly_white")
fig_log.update_xaxes(matches=None, title="")
fig_log.update_layout(height=380, showlegend=False)
fig_log.show()

**Interpretación.**

*   **Matrícula: el logaritmo funciona.** La asimetría baja a casi 0 y la distribución queda prácticamente simétrica.
    Además tiene una interpretación simple: una unidad de `log2` equivale a **duplicar** la matrícula, que es como se
    leyó el odds ratio en la sección 6.3. Entra a la matriz como `log_mat_total`.
*   **Tasas: el logaritmo ayuda poco.** La asimetría baja, pero sigue muy por encima de 1. El problema no es la cola
    larga sino la **masa de ceros**, y ninguna transformación puede separar colegios que tienen exactamente el mismo
    valor. Para estas variables la representación útil es la discretización de 7.2 (`registra_incidencia`), no el log.
    En la matriz se conservan en su escala original, que es la que se interpreta.

### <font color=skyblue> 7.4 Escalamiento </font>

Escalar pone las variables en una escala común, para que una variable medida en cientos (la matrícula) no pese más que
una medida de 0 a 3 (el índice) cuando se calculan distancias entre colegios. Se comparan los tres escaladores del curso:

| Escalador | Fórmula | Punto débil |
|---|---|---|
| Z-score (`StandardScaler`) | (x − media) / desviación | Sensible a valores extremos |
| Min-max (`MinMaxScaler`) | (x − mín) / (máx − mín) | Un solo valor extremo aplasta al resto |
| Robusto (`RobustScaler`) | (x − mediana) / IQR | No funciona si el IQR es 0 |

In [64]:
def resumen_escalado(serie, escalador):
    valores = escalador.fit_transform(serie.dropna().to_frame()).ravel()
    return pd.Series({"mediana": np.median(valores), "percentil_99": np.percentile(valores, 99),
                      "máximo": valores.max(), "asimetría": pd.Series(valores).skew()})

escaladores = {"z-score": StandardScaler(), "min-max": MinMaxScaler(), "robusto": RobustScaler()}

for variable in ["mat_total", "tasa_incidencias_100"]:
    print(f"--- {variable} ---")
    display(pd.DataFrame({nombre: resumen_escalado(base_final[variable], esc)
                          for nombre, esc in escaladores.items()}).T.round(3))

print("IQR de tasa_incidencias_100:", base_final["tasa_incidencias_100"].quantile(0.75) - base_final["tasa_incidencias_100"].quantile(0.25))

--- mat_total ---


,mediana,percentil_99,máximo,asimetría
z-score,-0.379,4.431,12.483,3.58
min-max,0.014,0.383,1.000,3.58
robusto,0.000,9.803,26.214,3.58


--- tasa_incidencias_100 ---


,mediana,percentil_99,máximo,asimetría
z-score,-0.201,4.218,30.129,11.031
min-max,0.000,0.146,1.000,11.031
robusto,0.000,14.568,100.000,11.031


IQR de tasa_incidencias_100: 0.0


**Interpretación.**

*   **Ningún escalador cambia la forma de la distribución:** la asimetría es la misma antes y después. Escalar cambia
    las unidades, no la forma. Por eso primero se aplica el logaritmo (7.3) y después se escala.
*   **Min-max no sirve con nuestros datos:** en la matrícula, el colegio más grande (más de 2,000 estudiantes) fija el
    máximo, y la mitad de los colegios queda comprimida por debajo de 0.02. Todos parecen iguales.
*   **El escalador robusto no hace nada con las tasas de incidencias:** su IQR es 0 (el mismo problema que impidió
    usar la regla del IQR para atípicos en 1.8), así que sklearn lo deja sin escalar y los valores siguen llegando a 100.
*   **Se usa z-score, y solo sobre variables sin masa de ceros:** `log_mat_total`, `indice_convivencia` y
    `pct_mujeres`. Aplicado a `tasa_incidencias_100` produciría valores de hasta z = 30 que dominarían cualquier
    distancia. Las columnas escaladas llevan el sufijo `_z` y se guardan **aparte** de las originales: las `_z` son
    para técnicas por distancia; las originales, para interpretar.

### <font color=skyblue> 7.5 Resumen de transformaciones </font>

In [65]:
resumen_transformaciones = pd.DataFrame([
    ["Codificación binaria", "7 preguntas SI/NO", "Sí", "1/0 conservando NaN; la media es directamente el % de SI"],
    ["One-hot", "Tipo de gestión (3 categorías)", "Sí", "Categorías sin orden; evita el orden falso del label encoding"],
    ["One-hot", "Área (2 categorías)", "Sí, como una columna", "Dos columnas serían redundantes: basta con urbano (0/1)"],
    ["One-hot", "Departamento (25 categorías)", "No", "25 columnas casi vacías; se deja como etiqueta y se controlará en el modelo"],
    ["Codificación ordinal", "Tamaño del colegio", "Sí", "Tiene orden natural (1 a 4)"],
    ["Exclusión", "Tipo de alumnado por sexo", "No", "Más del 99% son mixtos: casi constante"],
    ["Discretización por reglas", "Matrícula -> tam_colegio", "Sí", "Cortes con sentido educativo y grupos con suficientes colegios"],
    ["Discretización igual ancho", "Matrícula", "No", "Casi todos los colegios caen en el primer tramo"],
    ["Discretización por cuantiles", "Matrícula", "No", "Cortes que dependen de la muestra y no son interpretables"],
    ["Discretización por cuantiles", "Incidencias", "No", "Falla: los cuartiles valen 0"],
    ["Discretización por reglas", "Incidencias -> registra / nivel_registro", "Sí", "Separa colegios sin registro de los que registran"],
    ["Logaritmo", "Matrícula", "Sí", "La asimetría baja a casi 0; se interpreta como duplicar la matrícula"],
    ["Logaritmo (log1p)", "Tasas de incidencias, repetición y retiro", "No", "La asimetría sigue alta: el problema es la masa de ceros"],
    ["Z-score", "log_mat_total, indice_convivencia, pct_mujeres", "Sí", "Escala común para distancias, en columnas _z aparte"],
    ["Z-score", "Tasas", "No", "Valores de hasta z = 30 dominarían las distancias"],
    ["Min-max", "Matrícula", "No", "Un colegio extremo comprime al resto por debajo de 0.02"],
    ["Robusto", "Tasas de incidencias", "No", "IQR = 0: no escala nada"],
], columns=["transformación", "variable", "¿entra a la matriz?", "motivo"])

display(resumen_transformaciones)

,transformación,variable,¿entra a la matriz?,motivo
0,Codificación binaria,7 preguntas SI/NO,Sí,1/0 conservando NaN; la media es directamente ...
1,One-hot,Tipo de gestión (3 categorías),Sí,Categorías sin orden; evita el orden falso del...
2,One-hot,Área (2 categorías),"Sí, como una columna",Dos columnas serían redundantes: basta con urb...
3,One-hot,Departamento (25 categorías),No,25 columnas casi vacías; se deja como etiqueta...
4,Codificación ordinal,Tamaño del colegio,Sí,Tiene orden natural (1 a 4)
5,Exclusión,Tipo de alumnado por sexo,No,Más del 99% son mixtos: casi constante
6,Discretización por reglas,Matrícula -> tam_colegio,Sí,Cortes con sentido educativo y grupos con sufi...
7,Discretización igual ancho,Matrícula,No,Casi todos los colegios caen en el primer tramo
8,Discretización por cuantiles,Matrícula,No,Cortes que dependen de la muestra y no son int...
9,Discretización por cuantiles,Incidencias,No,Falla: los cuartiles valen 0


## <font color=yellow> 8. Matriz analítica </font>

Última etapa: reunir en una sola tabla las variables preparadas en la sección 7. **Cada fila sigue siendo un colegio de
primaria** (`COD_MOD` + `ANEXO`); lo que cambia es la representación de las variables.

In [66]:
# Identificadores (solo trazabilidad, no son características) y etiqueta para agrupar
matriz = base_final[["COD_MOD", "ANEXO", "D_DPTO"]].copy()

# Numéricas en su escala original (para interpretar)
numericas = ["mat_total", "pct_mujeres", "incidencias_libro", "casos_siseve",
             "tasa_incidencias_100", "tasa_casos_siseve_100", "tasa_repeticion",
             "tasa_retiro", "indice_convivencia"]
matriz[numericas] = base_final[numericas].astype(float)

# Derivadas y discretizadas (7.2 y 7.3)
matriz["log_mat_total"] = np.log(matriz["mat_total"])
matriz["tam_colegio_ord"] = tam_colegio_ord
matriz["registra_incidencia"] = registra_incidencia
matriz["nivel_registro"] = nivel_registro

# Codificadas (7.1)
matriz = matriz.join(codificadas)
matriz["urbano"] = urbano
matriz = matriz.join(dummies_gestion)

# Control de calidad: matrícula imputada por el MINEDU (permite excluirla en pruebas de robustez)
matriz["matricula_imputada"] = (base_final["IMPUTADO"] != "1_INFORMANTE").astype(int)

# Escaladas con z-score (7.4), en columnas nuevas
a_escalar = ["log_mat_total", "indice_convivencia", "pct_mujeres"]
matriz[[c + "_z" for c in a_escalar]] = StandardScaler().fit_transform(matriz[a_escalar])

print("Matriz analítica:", matriz.shape)
print("¿Una fila por colegio?", matriz.duplicated(subset=LLAVE).sum() == 0)
display(matriz.head())

Matriz analítica: (37354, 31)
¿Una fila por colegio? True


,COD_MOD,ANEXO,D_DPTO,mat_total,pct_mujeres,incidencias_libro,casos_siseve,tasa_incidencias_100,tasa_casos_siseve_100,tasa_repeticion,tasa_retiro,indice_convivencia,log_mat_total,tam_colegio_ord,registra_incidencia,nivel_registro,resp_convivencia,tiene_libro,conoce_protocolos,reporta_ugel,afiliado_siseve,actividad_esi,consulta_rnssc,urbano,gestion_privada,gestion_publica,gestion_publica_convenio,matricula_imputada,log_mat_total_z,indice_convivencia_z,pct_mujeres_z
0,1720556,0,AREQUIPA,300.0,0.5000,0.0,0.0,0.000,0.0,0.0000,0.0,3.0,5.703782,3,0.0,0.0,1.0,1.0,1.0,1.0,0.0,NaN,1.0,1,1,0,0,0,1.470232,0.506525,0.123163
1,0313890,0,AREQUIPA,29.0,0.6207,0.0,0.0,0.000,0.0,0.0000,0.0,3.0,3.367296,1,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0,0,1,0,0,-0.111906,0.506525,0.927767
2,1495092,0,HUANUCO,209.0,0.5024,1.0,0.0,0.478,0.0,0.0164,0.0,3.0,5.342334,3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1,0,1,0,0,1.225479,0.506525,0.139162
3,1579861,0,LIMA,127.0,0.4724,0.0,0.0,0.000,0.0,0.0000,0.0,2.0,4.844187,3,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1,1,0,0,0,0.888162,-0.865455,-0.060822
4,1529767,0,AREQUIPA,207.0,0.4686,0.0,0.0,0.000,0.0,0.0000,0.0,2.0,5.332719,3,0.0,0.0,1.0,1.0,0.0,1.0,1.0,NaN,0.0,1,1,0,0,0,1.218968,-0.865455,-0.086154


In [67]:
# Cuántos colegios quedan completos según lo que necesite el análisis siguiente
clave_registro = ["registra_incidencia", "log_mat_total", "indice_convivencia", "urbano"]
clave_trayectoria = clave_registro + ["tasa_repeticion", "tasa_retiro"]
for nombre, columnas in [("Análisis del registro", clave_registro),
                         ("Registro + trayectoria escolar", clave_trayectoria)]:
    completas = matriz.dropna(subset=columnas)
    print(f"{nombre}: {len(completas)} de {len(matriz)} colegios completos ({len(completas) / len(matriz):.1%})")

Análisis del registro: 31418 de 37354 colegios completos (84.1%)
Registro + trayectoria escolar: 31329 de 37354 colegios completos (83.9%)


La matriz conserva los NaN a propósito: eliminar filas es una decisión que depende de la técnica que se aplique
después, y el conteo anterior muestra cuánto costaría. Las tasas quedan en NaN en los colegios con menos de 10
estudiantes (sección 5.5), porque ahí una sola incidencia produce tasas que no son comparables.

### <font color=skyblue> 8.1 Diccionario de la matriz </font>

In [68]:
descripcion = {
    "COD_MOD": ("Identificador", "Código modular (llave, solo trazabilidad)"),
    "ANEXO": ("Identificador", "Anexo (llave, solo trazabilidad)"),
    "D_DPTO": ("Etiqueta", "Departamento, para agrupar resultados"),
    "mat_total": ("Numérica", "Matrícula total de primaria 2025"),
    "pct_mujeres": ("Numérica", "Proporción de mujeres matriculadas"),
    "incidencias_libro": ("Numérica", "Incidencias anotadas en el libro en 2024"),
    "casos_siseve": ("Numérica", "Casos reportados al SíseVe"),
    "tasa_incidencias_100": ("Derivada", "Incidencias por cada 100 estudiantes"),
    "tasa_casos_siseve_100": ("Derivada", "Casos SíseVe por cada 100 estudiantes"),
    "tasa_repeticion": ("Derivada", "Repitentes / evaluados"),
    "tasa_retiro": ("Derivada", "Retirados / evaluados"),
    "indice_convivencia": ("Derivada", "Índice de gestión de la convivencia (0 a 3)"),
    "log_mat_total": ("Transformada", "Logaritmo de la matrícula"),
    "tam_colegio_ord": ("Discretizada / ordinal", "Tamaño: 1 muy pequeño … 4 grande"),
    "registra_incidencia": ("Discretizada / binaria", "1 si registró al menos una incidencia"),
    "nivel_registro": ("Discretizada / ordinal", "0 no registra, 1 registra 1-2, 2 registra 3 o más"),
    "resp_convivencia": ("Binaria", "Tiene responsable de convivencia (P123B)"),
    "tiene_libro": ("Binaria", "Tiene libro de registro de incidencias (P125B)"),
    "conoce_protocolos": ("Binaria", "El director conoce los protocolos (P128B)"),
    "reporta_ugel": ("Binaria", "Sabe que debe reportar a la UGEL (P129B)"),
    "afiliado_siseve": ("Binaria", "Afiliado al SíseVe (P124B)"),
    "actividad_esi": ("Binaria", "Actividad de ESI con familias (P116A)"),
    "consulta_rnssc": ("Binaria", "Consulta el registro de sancionados (P127B)"),
    "urbano": ("Binaria", "1 si el área es urbana"),
    "gestion_privada": ("One-hot", "Gestión privada"),
    "gestion_publica": ("One-hot", "Pública de gestión directa"),
    "gestion_publica_convenio": ("One-hot", "Pública de gestión privada (convenio)"),
    "matricula_imputada": ("Binaria / control", "1 si el MINEDU imputó la matrícula"),
    "log_mat_total_z": ("Escalada", "Z-score de log_mat_total"),
    "indice_convivencia_z": ("Escalada", "Z-score del índice de convivencia"),
    "pct_mujeres_z": ("Escalada", "Z-score de pct_mujeres"),
}

diccionario = pd.DataFrame({
    "tipo": [descripcion[c][0] for c in matriz.columns],
    "descripción": [descripcion[c][1] for c in matriz.columns],
    "% faltantes": (matriz.isna().mean() * 100).round(1).values,
}, index=matriz.columns).rename_axis("variable")

display(diccionario)

,tipo,descripción,% faltantes
variable,,,
COD_MOD,Identificador,"Código modular (llave, solo trazabilidad)",0.0
ANEXO,Identificador,"Anexo (llave, solo trazabilidad)",0.0
D_DPTO,Etiqueta,"Departamento, para agrupar resultados",0.0
mat_total,Numérica,Matrícula total de primaria 2025,0.0
pct_mujeres,Numérica,Proporción de mujeres matriculadas,0.0
incidencias_libro,Numérica,Incidencias anotadas en el libro en 2024,15.8
casos_siseve,Numérica,Casos reportados al SíseVe,0.1
tasa_incidencias_100,Derivada,Incidencias por cada 100 estudiantes,31.5
tasa_casos_siseve_100,Derivada,Casos SíseVe por cada 100 estudiantes,20.5


### <font color=skyblue> 8.2 Exclusiones importantes </font>

| Variable(s) excluida(s) | Motivo |
|---|---|
| `CEN_EDU`, `DISTRITO`, `LOCALIDAD` | Texto libre de identificación; no es una característica del colegio |
| `NLAT_IE`, `NLONG_IE` | No se hará análisis espacial en esta etapa; el departamento resume la ubicación |
| `D_PROV`, `D_DIST`, `D_DREUGEL` | Demasiadas categorías (225 UGEL) para una primera matriz |
| `mat_g1` … `mat_g6_m` | Redundantes: se resumen en `mat_total` y `pct_mujeres` |
| `res_*` (conteos del resultado) | Se resumen en `tasa_repeticion` y `tasa_retiro` |
| `motret_violencia` | Solo 6 estudiantes en 5 colegios de todo el país (sección 2.6) |
| `D_TIPSSEXO` | Más del 99% de colegios mixtos: casi constante (7.1) |
| `P124B_NO`, `P121A` | Motivo de no afiliación y criterio de secciones: códigos nominales sin relación directa con el objetivo |
| `P112A`–`P114A`, `P103B`–`P106B` | Revisadas en la limpieza; se dejan fuera para mantener la matriz enfocada |
| `tasa_recup_ped` | Recuperación pedagógica: fuera del objetivo planteado |
| One-hot de departamento | 25 columnas casi vacías (7.1) |

In [69]:
matriz.to_csv(SALIDAS / "matriz_analitica.csv", index=False)
diccionario.to_csv(SALIDAS / "diccionario_matriz.csv")
print("Guardados:", SALIDAS / "matriz_analitica.csv", "y", SALIDAS / "diccionario_matriz.csv")

Guardados: salidas/matriz_analitica.csv y salidas/diccionario_matriz.csv


## <font color=yellow> 9. Próximo paso </font>

No se implementa en este hito; se deja planteado.

1.  **Modelo de registro con control regional.** Reestimar la regresión logística de `registra_incidencia` agregando el
    departamento (`C(D_DPTO)`), que es la principal limitación del modelo actual, y comparar con y sin los colegios con
    matrícula imputada (`matricula_imputada`).
2.  **Perfiles de colegios (K-means).** Agrupar colegios con las columnas `_z` y las binarias de gestión de la
    convivencia, para encontrar perfiles de “capacidad de registro” que la UGEL pueda priorizar.

**Qué falta validar:** el efecto de los faltantes no aleatorios (Loreto y Ucayali), la sensibilidad de las tasas al
umbral de 10 estudiantes y el desfase entre incidencias de 2024 y matrícula de 2025.

## <font color=yellow> 10. Conclusiones </font>

**Sobre el problema planteado**

1.  **El registro de violencia escolar depende más de la capacidad y el tamaño del colegio que de
    la violencia misma.** Entre los colegios con índice de gestión 0, el 3% registra incidencias;
    entre los de índice 3, el 15%. Por tamaño, el registro va de 5% en colegios muy pequeños a 45%
    en los grandes. Un colegio que reporta cero casos y no tiene responsable de convivencia no es un
    colegio sin violencia: es un colegio sin registro.

2.  **Los factores institucionales asociados al registro, con los demás constantes, son el tamaño,
    tener un responsable de convivencia, conocer los protocolos y hacer actividades de ESI con
    familias.** Los odds ratio son 1.6 por cada duplicación de la matrícula, 1.6, 1.4 y 1.3
    respectivamente. La afiliación al SíseVe no aporta por sí sola.

3.  **Existe una brecha consistente entre el registro interno y el reporte al SíseVe.** En los 25
    departamentos se anotan más incidencias en el libro de registro que casos en el portal
    (razón de 2.6 a 1), lo que sugiere que el dato del SíseVe subestima la magnitud del fenómeno.

4.  **Las tasas más altas no están donde están los casos absolutos.** Lima ocupa el puesto 1 por
    número de incidencias y el 16 por tasa; las tasas más altas se observan en Pasco, Loreto,
    Ucayali y Amazonas, aunque en Loreto y Ucayali entre un tercio y casi la mitad de los colegios
    no respondió.

5.  **La repetición muestra brechas claras por gestión y área, y está muy concentrada.** La tasa
    agregada va de 0.15% en la privada urbana a 2.1% en la pública rural, y el 4.6% de los colegios
    acumula la mitad de los repitentes.

6.  **La relación entre violencia registrada y trayectoria escolar es muy débil.** La correlación
    de Spearman es de 0.06 con la repetición y 0.09 con el retiro, y cercana a cero dentro de cada
    grupo de gestión y área. Con los datos disponibles no se puede afirmar que los colegios con más
    violencia registrada tengan peores resultados académicos.

**Sobre el tratamiento de los datos**

7.  El cambio de la base de denuncias del SíseVe por fuentes con código modular fue la
    decisión metodológica central del trabajo. Permitió pasar de un análisis a nivel UGEL
    disfrazado de análisis a nivel colegio, a una integración 1 a 1 real entre las cuatro
    fuentes, sin duplicación de llaves.

8.  El tratamiento de las preguntas condicionadas (`P124B_SI`, `P126B_SI_1`) resultó ser
    determinante. Distinguir entre "respondió cero", "no corresponde responder" y "no
    respondió" cambia por completo la magnitud de los indicadores de violencia.

9.  La elección del estadístico importó tanto como la limpieza. Con distribuciones
    concentradas en cero y colegios de tamaños muy dispares, el promedio de tasas
    individuales y la correlación de Pearson resultaron engañosos; la tasa agregada, la
    proporción de colegios que registran y la correlación de Spearman dieron lecturas estables.

10. Tres decisiones de diseño evitaron sesgos: la matrícula se une por separado para que ningún
    colegio pierda su denominador; el índice de gestión excluye el libro de incidencias para no ser
    circular; y los faltantes en incidencias, que no son aleatorios, se documentaron en lugar de imputarse.

11. **Las transformaciones se aplicaron solo donde servían.** El logaritmo corrigió la asimetría de la matrícula
    (de 3.6 a 0.1), pero no la de las tasas, cuyo problema es la masa de ceros; por eso la incidencia se
    discretizó (`registra_incidencia`) en lugar de escalarse. Min-max, el escalador robusto, los cuantiles y el
    one-hot del departamento se probaron y se descartaron con evidencia (sección 7.5).

12. La base integrada se transformó en una matriz analítica de 37,354 colegios y 31 columnas (sección 8), con
    diccionario de variables y exclusiones documentadas.

**Limitaciones**

13. Ambos indicadores de violencia son **autodeclarados por el director** y están mediados
    por la capacidad de registro del colegio, por lo que miden violencia *registrada*, no
    violencia *ocurrida*. Esta es la limitación de fondo del trabajo.
14. El 16% de los colegios no tiene dato de incidencias y la falta de respuesta es mayor en colegios
    pequeños, rurales y de la Amazonía. Las comparaciones entre grupos se calculan solo con quienes
    respondieron.
15. El motivo de retiro por violencia, que sería la medida más directa, resultó
    inutilizable: solo 6 estudiantes en 5 colegios de todo el país.
16. Parte de los datos de matrícula fueron imputados por el MINEDU (variable `IMPUTADO`, 3.3% de los
    colegios) y no declarados directamente por el colegio.
17. El análisis es transversal. Las incidencias corresponden a 2024 y la repetición y la matrícula a
    2025, así que no es un panel ni permite afirmar causalidad en ninguna dirección; además, la
    matrícula de 2025 se usa como denominador de incidencias de 2024.
18. La regresión logística no controla por departamento, por lo que parte de las diferencias
    regionales puede estar dentro de los coeficientes.